# NFL Big Data Bowl 2027 -- reproducible analysis

This notebook reproduces the headline result of the accompanying writeup from the raw competition CSVs. It uses the 10 Hz **Combine tracking** (`combine_tracking.csv`), the 10 Hz **in-game tracking** (`game_tracking_YYYY.csv`) and `player_play.csv`.

**Claim under test:** `skill_dl__entry_speed_max` predicts `sack_rate` for the DL group, over and above the conventional Combine battery and draft capital.

**Inputs.** Only the competition dataset `nfl-big-data-bowl-2027` is required. The analysis package is inlined in the next cell, so no extra dataset and no path editing is needed.


In [ ]:
# ---------------------------------------------------------------------------
# The analysis package, inlined so this notebook is self-contained.
# It is the same code as solutions/nfl_big_data_bowl_2027 in the repository.
# ---------------------------------------------------------------------------
import json as _json, sys, types

_BDB_PAYLOAD = '{"paths": "\\"\\"\\"Filesystem layout for the NFL Big Data Bowl 2027 solution package.\\n\\nAll heavy CSV inputs are read from ``<DATA_DIR>`` and every derived artefact is\\nwritten under ``<ARTIFACT_DIR>`` so that the pipeline is fully re-runnable and\\nnothing large is ever committed to git.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport os\\nfrom pathlib import Path\\n\\nPACKAGE_ROOT = Path(__file__).resolve().parent.parent\\nREPO_ROOT = PACKAGE_ROOT.parent.parent\\n\\n# ---------------------------------------------------------------------------\\n# Input data\\n# ---------------------------------------------------------------------------\\n# Order of preference:\\n#   1. $BDB_DATA_DIR\\n#   2. solutions/nfl_big_data_bowl_2027/data\\n#   3. multi_agents/competition/nfl_big_data_bowl_2027/data\\n#   4. a Kaggle input mount (so the public notebook runs unchanged on Kaggle)\\n_CANDIDATE_DATA_DIRS = [\\n    PACKAGE_ROOT / \\"data\\",\\n    REPO_ROOT / \\"multi_agents\\" / \\"competition\\" / \\"nfl_big_data_bowl_2027\\" / \\"data\\",\\n]\\n\\n# Kaggle mounts competition data read-only under /kaggle/input/<slug>. Scanning\\n# for it means the published notebook needs no path edit when it is re-run there.\\nKAGGLE_INPUT_ROOT = Path(\\"/kaggle/input\\")\\nKAGGLE_SLUG = \\"nfl-big-data-bowl-2027\\"\\n\\n\\ndef _kaggle_data_dirs() -> list:\\n    \\"\\"\\"Directories under /kaggle/input that actually hold the competition CSVs.\\"\\"\\"\\n    if not KAGGLE_INPUT_ROOT.is_dir():\\n        return []\\n    # Literals, not the module constants: this runs during module import, before\\n    # the file-name section below has been executed.\\n    probe = (\\"players.csv\\", \\"combine_tracking.csv\\")\\n    found = []\\n    preferred = KAGGLE_INPUT_ROOT / KAGGLE_SLUG\\n    # depth 2 covers both /kaggle/input/<slug>/ and /kaggle/input/<dataset>/<slug>/\\n    for candidate in [preferred, *sorted(KAGGLE_INPUT_ROOT.glob(\\"*\\")),\\n                      *sorted(KAGGLE_INPUT_ROOT.glob(\\"*/*\\"))]:\\n        if candidate.is_dir() and all((candidate / name).exists() for name in probe):\\n            if candidate not in found:\\n                found.append(candidate)\\n    return found\\n\\n\\ndef data_dir() -> Path:\\n    override = os.environ.get(\\"BDB_DATA_DIR\\")\\n    if override:\\n        return Path(override).expanduser().resolve()\\n    for candidate in _CANDIDATE_DATA_DIRS:\\n        if candidate.is_dir():\\n            return candidate\\n    kaggle = _kaggle_data_dirs()\\n    if kaggle:\\n        return kaggle[0]\\n    return _CANDIDATE_DATA_DIRS[0]\\n\\n\\nDATA_DIR = data_dir()\\n\\n# ---------------------------------------------------------------------------\\n# Outputs\\n# ---------------------------------------------------------------------------\\nARTIFACT_DIR = Path(os.environ.get(\\"BDB_ARTIFACT_DIR\\", PACKAGE_ROOT / \\"artifacts\\"))\\nCACHE_DIR = Path(os.environ.get(\\"BDB_CACHE_DIR\\", PACKAGE_ROOT / \\"cache\\"))\\nFIGURE_DIR = ARTIFACT_DIR / \\"figures\\"\\nREPORT_DIR = ARTIFACT_DIR / \\"reports\\"\\n\\n# ---------------------------------------------------------------------------\\n# Input file names\\n# ---------------------------------------------------------------------------\\nPLAYERS = \\"players.csv\\"\\nCOMBINE_RESULTS = \\"combine_results.csv\\"\\nCOMBINE_TRACKING = \\"combine_tracking.csv\\"\\nCAREER_SUCCESSES = \\"player_career_successes.csv\\"\\nPLAYER_PLAY = \\"player_play.csv\\"\\nGAMES = \\"games.csv\\"\\nGAME_TRACKING = {\\n    2023: \\"game_tracking_2023.csv\\",\\n    2024: \\"game_tracking_2024.csv\\",\\n    2025: \\"game_tracking_2025.csv\\",\\n}\\n\\nREQUIRED_FILES = [\\n    PLAYERS,\\n    COMBINE_RESULTS,\\n    COMBINE_TRACKING,\\n    CAREER_SUCCESSES,\\n    PLAYER_PLAY,\\n    GAMES,\\n    *GAME_TRACKING.values(),\\n]\\n\\n# Derived caches\\nCACHE_COMBINE_ATTEMPT_FEATURES = \\"combine_attempt_features.parquet\\"\\nCACHE_COMBINE_PLAYER_FEATURES = \\"combine_player_features.parquet\\"\\nCACHE_INGAME_PLAY_FEATURES = \\"ingame_play_features.parquet\\"\\nCACHE_INGAME_PLAYER_FEATURES = \\"ingame_player_features.parquet\\"\\nCACHE_PLAYER_PLAY_AGG = \\"player_play_agg.parquet\\"\\nCACHE_ANALYSIS_TABLE = \\"analysis_table.parquet\\"\\n\\nKAGGLE_COMPETITION = \\"nfl-big-data-bowl-2027\\"\\n\\n\\ndef ensure_dirs() -> None:\\n    for path in (DATA_DIR, ARTIFACT_DIR, CACHE_DIR, FIGURE_DIR, REPORT_DIR):\\n        path.mkdir(parents=True, exist_ok=True)\\n\\n\\ndef missing_files() -> list[str]:\\n    return [name for name in REQUIRED_FILES if not (DATA_DIR / name).exists()]\\n\\n\\ndef describe() -> str:\\n    lines = [\\n        f\\"PACKAGE_ROOT   : {PACKAGE_ROOT}\\",\\n        f\\"DATA_DIR       : {DATA_DIR}\\",\\n        f\\"ARTIFACT_DIR   : {ARTIFACT_DIR}\\",\\n        f\\"CACHE_DIR      : {CACHE_DIR}\\",\\n    ]\\n    missing = missing_files()\\n    if missing:\\n        lines.append(f\\"MISSING ({len(missing)}): \\" + \\", \\".join(missing))\\n    else:\\n        lines.append(\\"All required input files present.\\")\\n    return \\"\\\\n\\".join(lines)\\n", "schema": "\\"\\"\\"Column contracts for the nine NFL Big Data Bowl 2027 source tables.\\n\\nThe schema is declared explicitly (rather than inferred) so that the pipeline\\nfails loudly -- instead of silently -- if Kaggle ever ships a different header.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Dict, List\\n\\n# ---------------------------------------------------------------------------\\n# players.csv\\n# ---------------------------------------------------------------------------\\nPLAYERS_COLUMNS: List[str] = [\\n    \\"nfl_id\\",\\n    \\"display_name\\",\\n    \\"draft_year\\",\\n    \\"nfl_position\\",\\n    \\"birth_date\\",\\n    \\"college_name\\",\\n    \\"college_conference\\",\\n    \\"draft_round\\",\\n    \\"draft_pick_within_round\\",\\n    \\"draft_overall_pick\\",\\n]\\n\\nPLAYERS_DTYPES: Dict[str, str] = {\\n    \\"nfl_id\\": \\"int64\\",\\n    \\"display_name\\": \\"string\\",\\n    \\"draft_year\\": \\"int64\\",\\n    \\"nfl_position\\": \\"string\\",\\n    \\"birth_date\\": \\"string\\",\\n    \\"college_name\\": \\"string\\",\\n    \\"college_conference\\": \\"string\\",\\n    \\"draft_round\\": \\"Float64\\",\\n    \\"draft_pick_within_round\\": \\"Float64\\",\\n    \\"draft_overall_pick\\": \\"Float64\\",\\n}\\n\\n# ---------------------------------------------------------------------------\\n# combine_results.csv\\n# ---------------------------------------------------------------------------\\nCOMBINE_RESULTS_COLUMNS: List[str] = [\\n    \\"draft_year\\",\\n    \\"nfl_id\\",\\n    \\"combine_position\\",\\n    \\"combine_height\\",\\n    \\"combine_weight\\",\\n    \\"hand_size\\",\\n    \\"arm_length\\",\\n    \\"wing_span\\",\\n    \\"ten_yd_split\\",\\n    \\"forty\\",\\n    \\"vertical\\",\\n    \\"broad_jump\\",\\n    \\"three_cone\\",\\n    \\"short_shuttle\\",\\n    \\"bench_reps\\",\\n    \\"ngs_athleticism_score\\",\\n    \\"ngs_college_production_score\\",\\n    \\"ngs_final_score\\",\\n]\\n\\n# Traditional (stopwatch / tape) testing numbers. These form the *baseline*\\n# that every sensor-derived feature must beat to be interesting.\\nTRADITIONAL_COMBINE_METRICS: List[str] = [\\n    \\"forty\\",\\n    \\"ten_yd_split\\",\\n    \\"three_cone\\",\\n    \\"short_shuttle\\",\\n    \\"vertical\\",\\n    \\"broad_jump\\",\\n    \\"bench_reps\\",\\n]\\n\\nANTHROPOMETRIC_METRICS: List[str] = [\\n    \\"combine_height\\",\\n    \\"combine_weight\\",\\n    \\"hand_size\\",\\n    \\"arm_length\\",\\n    \\"wing_span\\",\\n]\\n\\n# ---------------------------------------------------------------------------\\n# combine_tracking.csv  (10 Hz optical tracking during Combine drills)\\n# ---------------------------------------------------------------------------\\nCOMBINE_TRACKING_COLUMNS: List[str] = [\\n    \\"draft_year\\",\\n    \\"event_id\\",\\n    \\"nfl_id\\",\\n    \\"entity_type\\",\\n    \\"time\\",\\n    \\"drill_type\\",\\n    \\"drill_name\\",\\n    \\"attempt\\",\\n    \\"x\\",\\n    \\"y\\",\\n    \\"s\\",\\n    \\"a\\",\\n    \\"dis\\",\\n    \\"dir\\",\\n]\\n\\nCOMBINE_TRACKING_DTYPES: Dict[str, str] = {\\n    \\"draft_year\\": \\"int64\\",\\n    \\"event_id\\": \\"string\\",\\n    \\"nfl_id\\": \\"Int64\\",\\n    \\"entity_type\\": \\"string\\",\\n    \\"drill_type\\": \\"string\\",\\n    \\"drill_name\\": \\"string\\",\\n    \\"attempt\\": \\"Int64\\",\\n    \\"x\\": \\"float64\\",\\n    \\"y\\": \\"float64\\",\\n    \\"s\\": \\"float64\\",\\n    \\"a\\": \\"float64\\",\\n    \\"dis\\": \\"float64\\",\\n    \\"dir\\": \\"float64\\",\\n}\\n\\nFORTY_YARD_DASH = \\"FORTY_YARD_DASH\\"\\nTHREE_CONE_DRILL = \\"THREE_CONE_DRILL\\"\\nSHORT_SHUTTLE = \\"SHORT_SHUTTLE\\"\\nFORTY_DRILL_TYPES = [FORTY_YARD_DASH]\\n\\n# Traits that only carry meaning when the drill actually contains a change of\\n# direction. A 40-yard dash is a straight line: any \\"cut\\" the detector finds in\\n# it is tracking noise, and letting those features into the screen produces\\n# large spurious effects (verified: a noise \\"regain slope\\" reached the top of\\n# the real-data screen before this guard was added).\\nCOD_ONLY_FEATURES = {\\n    \\"speed_retention\\", \\"speed_loss\\", \\"entry_speed\\", \\"exit_speed\\",\\n    \\"cut_min_speed\\", \\"brake_rate\\", \\"exit_accel_rate\\", \\"regain_time_90\\",\\n    \\"regain_speed_slope\\", \\"turn_angle_deg\\", \\"turn_duration_s\\",\\n    \\"turn_rate_deg_s\\", \\"cut_radius_proxy\\", \\"turn_start_frac\\", \\"turn_sign\\",\\n    \\"n_turns\\", \\"max_turn_rate_deg_s\\", \\"total_abs_turn_deg\\",\\n}\\n\\n# Bookkeeping columns that must never be treated as sensor traits.\\nNON_TRAIT_FEATURES = {\\"mean_dt\\", \\"n_frames\\"}\\nCOD_DRILL_TYPES = [THREE_CONE_DRILL, SHORT_SHUTTLE]\\n# Prefixed \\"SKILL_DRILLS_<POSGROUP>\\"\\nSKILL_DRILL_PREFIX = \\"SKILL_DRILLS_\\"\\n\\n# ---------------------------------------------------------------------------\\n# player_career_successes.csv\\n# ---------------------------------------------------------------------------\\nCAREER_SUCCESS_COLUMNS: List[str] = [\\n    \\"nfl_id\\",\\n    \\"career_offensive_snaps\\",\\n    \\"career_defensive_snaps\\",\\n    \\"career_special_teams_snaps\\",\\n    \\"career_games_active\\",\\n    \\"career_games_started\\",\\n    \\"ap_all_pro_1st_team\\",\\n    \\"ap_all_pro_2nd_team\\",\\n    \\"pro_bowl_original_ballot\\",\\n]\\n\\n# ---------------------------------------------------------------------------\\n# player_play.csv -- in-game Next Gen Stats outcomes\\n# ---------------------------------------------------------------------------\\nPLAYER_PLAY_ID_COLUMNS: List[str] = [\\n    \\"game_id\\",\\n    \\"play_id\\",\\n    \\"team_abbr\\",\\n    \\"nfl_id\\",\\n    \\"lined_up_position\\",\\n]\\n\\n# Rate / efficiency outcomes that do NOT mechanically scale with opportunity.\\nPLAYER_PLAY_RATE_METRICS: List[str] = [\\n    \\"separation_at_pass_forward\\",\\n    \\"cushion\\",\\n    \\"player_get_off\\",\\n    \\"time_to_pressure\\",\\n    \\"time_to_qb_hurry\\",\\n    \\"peak_pressure_probability_allowed\\",\\n    \\"time_to_pressure_allowed\\",\\n    \\"dropback_duration\\",\\n    \\"pass_rushers_encountered\\",\\n    \\"expected_yards_after_catch\\",\\n    \\"yards_after_catch\\",\\n    \\"pass_length\\",\\n    \\"expected_points_added\\",\\n]\\n\\n# Binary / share outcomes (aggregated as means -> rates)\\nPLAYER_PLAY_BINARY_METRICS: List[str] = [\\n    \\"target\\",\\n    \\"sack\\",\\n    \\"tackle\\",\\n    \\"assist\\",\\n    \\"tackle_for_loss\\",\\n    \\"caused_forced_fumble\\",\\n    \\"recovered_fumble\\",\\n    \\"fumble\\",\\n    \\"fumble_lost\\",\\n    \\"pressure_allowed\\",\\n    \\"blitzing\\",\\n    \\"quick_pressure\\",\\n]\\n\\n# ---------------------------------------------------------------------------\\n# games.csv\\n# ---------------------------------------------------------------------------\\nGAMES_COLUMNS: List[str] = [\\n    \\"game_id\\",\\n    \\"game_key\\",\\n    \\"season\\",\\n    \\"season_type\\",\\n    \\"week\\",\\n    \\"home_team_abbr\\",\\n    \\"visitor_team_abbr\\",\\n]\\n\\n# ---------------------------------------------------------------------------\\n# game_tracking_YYYY.csv  (10 Hz in-game optical tracking)\\n# ---------------------------------------------------------------------------\\nGAME_TRACKING_COLUMNS: List[str] = [\\n    \\"game_id\\",\\n    \\"play_id\\",\\n    \\"nfl_id\\",\\n    \\"time\\",\\n    \\"x\\",\\n    \\"y\\",\\n    \\"s\\",\\n    \\"a\\",\\n    \\"dis\\",\\n    \\"o\\",\\n    \\"dir\\",\\n    \\"event\\",\\n]\\n\\nGAME_TRACKING_DTYPES: Dict[str, str] = {\\n    \\"game_id\\": \\"int64\\",\\n    \\"play_id\\": \\"int64\\",\\n    \\"nfl_id\\": \\"int64\\",\\n    \\"x\\": \\"float64\\",\\n    \\"y\\": \\"float64\\",\\n    \\"s\\": \\"float64\\",\\n    \\"a\\": \\"float64\\",\\n    \\"dis\\": \\"float64\\",\\n    \\"o\\": \\"float64\\",\\n    \\"dir\\": \\"float64\\",\\n    \\"event\\": \\"string\\",\\n}\\n\\n# ---------------------------------------------------------------------------\\n# Position grouping\\n# ---------------------------------------------------------------------------\\n# The competition focuses on five position groups.\\nPOSITION_GROUP: Dict[str, str] = {\\n    # wide receivers\\n    \\"WR\\": \\"WR\\",\\n    # tight ends\\n    \\"TE\\": \\"TE\\",\\n    # defensive backs\\n    \\"CB\\": \\"DB\\",\\n    \\"DB\\": \\"DB\\",\\n    \\"SS\\": \\"DB\\",\\n    \\"FS\\": \\"DB\\",\\n    \\"S\\": \\"DB\\",\\n    # offensive line\\n    \\"T\\": \\"OL\\",\\n    \\"G\\": \\"OL\\",\\n    \\"C\\": \\"OL\\",\\n    \\"OT\\": \\"OL\\",\\n    \\"OG\\": \\"OL\\",\\n    \\"OL\\": \\"OL\\",\\n    # defensive line\\n    \\"DE\\": \\"DL\\",\\n    \\"DT\\": \\"DL\\",\\n    \\"NT\\": \\"DL\\",\\n    \\"DL\\": \\"DL\\",\\n    # edge / off-ball linebackers are out of scope but kept for completeness\\n    \\"OLB\\": \\"LB\\",\\n    \\"ILB\\": \\"LB\\",\\n    \\"MLB\\": \\"LB\\",\\n    \\"LB\\": \\"LB\\",\\n    \\"RB\\": \\"RB\\",\\n    \\"FB\\": \\"RB\\",\\n}\\n\\nFOCUS_POSITION_GROUPS: List[str] = [\\"WR\\", \\"TE\\", \\"DB\\", \\"OL\\", \\"DL\\"]\\n\\n# Which in-game outcome families are meaningful for which position group.\\n# Used to bound the feasibility scan to hypotheses that a football analyst\\n# would consider legitimate (this is the analyst-in-the-loop prior).\\nPOSITION_TO_OUTCOMES: Dict[str, List[str]] = {\\n    \\"WR\\": [\\n        \\"sep_mean\\", \\"sep_p75\\", \\"cushion_mean\\", \\"target_rate\\",\\n        \\"yac_mean\\", \\"eyac_mean\\", \\"rec_yards_per_target\\",\\n    ],\\n    \\"TE\\": [\\n        \\"sep_mean\\", \\"sep_p75\\", \\"cushion_mean\\", \\"target_rate\\", \\"yac_mean\\",\\n    ],\\n    \\"DB\\": [\\n        \\"cushion_mean\\", \\"sep_allowed_mean\\", \\"tackle_rate\\", \\"assist_rate\\",\\n        \\"epa_allowed_mean\\",\\n    ],\\n    \\"OL\\": [\\n        \\"pressure_allowed_rate\\", \\"sack_allowed_rate\\", \\"ttp_allowed_mean\\",\\n        \\"peak_pressure_prob_allowed\\", \\"dropback_duration_mean\\",\\n    ],\\n    \\"DL\\": [\\n        \\"get_off_mean\\", \\"get_off_p25\\", \\"pressure_rate\\", \\"sack_rate\\",\\n        \\"tfl_rate\\", \\"quick_pressure_rate\\", \\"time_to_pressure_mean\\",\\n    ],\\n}\\n\\n# Which combine drill families are meaningful for which position group.\\nPOSITION_TO_DRILLS: Dict[str, List[str]] = {\\n    \\"WR\\": [FORTY_YARD_DASH, SHORT_SHUTTLE, THREE_CONE_DRILL, \\"SKILL_DRILLS_WR\\"],\\n    \\"TE\\": [FORTY_YARD_DASH, SHORT_SHUTTLE, THREE_CONE_DRILL, \\"SKILL_DRILLS_TE\\"],\\n    \\"DB\\": [FORTY_YARD_DASH, SHORT_SHUTTLE, THREE_CONE_DRILL, \\"SKILL_DRILLS_DB\\"],\\n    \\"OL\\": [FORTY_YARD_DASH, SHORT_SHUTTLE, THREE_CONE_DRILL, \\"SKILL_DRILLS_OL\\"],\\n    \\"DL\\": [FORTY_YARD_DASH, SHORT_SHUTTLE, THREE_CONE_DRILL, \\"SKILL_DRILLS_DL\\"],\\n}\\n", "io": "\\"\\"\\"Cached, validated loaders for the NFL Big Data Bowl 2027 source tables.\\n\\nDesign goals\\n------------\\n* Fail loudly on schema drift (header check against :mod:`bdb.schema`).\\n* Never hold more than one game-tracking season in memory at a time.\\n* Cache everything expensive as parquet so re-runs are cheap.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom pathlib import Path\\nfrom typing import Callable, Dict, Iterable, List, Optional\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom . import paths, schema\\n\\nlog = logging.getLogger(__name__)\\n\\n\\nclass SchemaError(RuntimeError):\\n    \\"\\"\\"Raised when a source file\'s header does not match the declared contract.\\"\\"\\"\\n\\n\\n# ---------------------------------------------------------------------------\\n# Low level helpers\\n# ---------------------------------------------------------------------------\\ndef _validate_header(\\n    path: Path,\\n    expected: Iterable[str],\\n    *,\\n    allow_extra: bool = False,\\n    sample_rows: int = 5,\\n) -> None:\\n    \\"\\"\\"Read only the header (+ a few rows) and verify the declared contract.\\"\\"\\"\\n    expected_set = set(expected)\\n    actual = pd.read_csv(path, nrows=sample_rows)\\n    actual_set = set(actual.columns)\\n    missing = expected_set - actual_set\\n    if missing:\\n        raise SchemaError(\\n            f\\"{path.name}: missing expected columns {sorted(missing)}; \\"\\n            f\\"got {sorted(actual_set)}\\"\\n        )\\n    if not allow_extra:\\n        extra = actual_set - expected_set\\n        if extra:\\n            log.warning(\\"%s: unexpected extra columns %s\\", path.name, sorted(extra))\\n\\n\\ndef read_csv(\\n    name: str,\\n    dtypes: Optional[Dict[str, str]] = None,\\n    *,\\n    usecols: Optional[List[str]] = None,\\n    validate_against: Optional[Iterable[str]] = None,\\n    **kwargs,\\n) -> pd.DataFrame:\\n    path = paths.DATA_DIR / name\\n    if not path.exists():\\n        raise FileNotFoundError(\\n            f\\"Missing competition file: {path}\\\\n\\"\\n            \\"Run `python -m bdb.cli download` (requires ~/.kaggle/kaggle.json and \\"\\n            \\"accepted competition rules) or place the file manually.\\"\\n        )\\n    kwargs.setdefault(\\"low_memory\\", False)\\n    if validate_against is not None:\\n        _validate_header(path, validate_against)\\n    return pd.read_csv(path, dtype=dtypes, usecols=usecols, **kwargs)\\n\\n\\ndef _csv_twin(cache_name: str) -> Path:\\n    \\"\\"\\"CSV twin of a ``*.parquet`` cache name.\\"\\"\\"\\n    return paths.CACHE_DIR / (Path(cache_name).stem + \\".csv\\")\\n\\n\\ndef read_parquet_cache(cache_name: str) -> Optional[pd.DataFrame]:\\n    \\"\\"\\"Read a cache, preferring parquet and falling back to the CSV twin.\\n\\n    ``pyarrow`` is optional here so the pipeline still runs where it cannot be\\n    installed (for example a Kaggle notebook without that wheel).\\n    \\"\\"\\"\\n    path = paths.CACHE_DIR / cache_name\\n    if path.exists():\\n        try:\\n            frame = pd.read_parquet(path)\\n            log.info(\\"cache hit: %s\\", path)\\n            return frame\\n        except Exception as exc:  # noqa: BLE001\\n            log.info(\\"parquet cache %s unreadable (%s); trying CSV twin\\", path.name, exc)\\n    twin = _csv_twin(cache_name)\\n    if twin.exists():\\n        log.info(\\"cache hit: %s\\", twin)\\n        return pd.read_csv(twin)\\n    return None\\n\\n\\ndef write_parquet_cache(df: pd.DataFrame, cache_name: str) -> None:\\n    \\"\\"\\"Write a cache as parquet when possible, always leaving a CSV twin.\\"\\"\\"\\n    paths.CACHE_DIR.mkdir(parents=True, exist_ok=True)\\n    path = paths.CACHE_DIR / cache_name\\n    try:\\n        df.to_parquet(path, index=False)\\n        log.info(\\"wrote cache: %s (%s rows)\\", path, len(df))\\n    except Exception as exc:  # noqa: BLE001\\n        log.warning(\\"parquet unavailable (%s); writing CSV twin only\\", exc)\\n    twin = _csv_twin(cache_name)\\n    df.to_csv(twin, index=False)\\n    log.info(\\"wrote cache: %s (%s rows)\\", twin, len(df))\\n\\n\\ndef cached(cache_name: str, builder: Callable[[], pd.DataFrame], *, force: bool = False) -> pd.DataFrame:\\n    \\"\\"\\"Memoise an expensive dataframe builder into the on-disk cache.\\"\\"\\"\\n    if not force:\\n        hit = read_parquet_cache(cache_name)\\n        if hit is not None:\\n            return hit\\n    df = builder()\\n    write_parquet_cache(df, cache_name)\\n    return df\\n\\n\\n# ---------------------------------------------------------------------------\\n# Individual tables\\n# ---------------------------------------------------------------------------\\ndef load_players() -> pd.DataFrame:\\n    return read_csv(\\n        paths.PLAYERS,\\n        schema.PLAYERS_DTYPES,\\n        validate_against=schema.PLAYERS_COLUMNS,\\n    )\\n\\n\\ndef load_combine_results() -> pd.DataFrame:\\n    return read_csv(\\n        paths.COMBINE_RESULTS,\\n        validate_against=schema.COMBINE_RESULTS_COLUMNS,\\n    )\\n\\n\\ndef load_career_successes() -> pd.DataFrame:\\n    return read_csv(\\n        paths.CAREER_SUCCESSES,\\n        validate_against=schema.CAREER_SUCCESS_COLUMNS,\\n    )\\n\\n\\ndef load_games() -> pd.DataFrame:\\n    return read_csv(paths.GAMES, validate_against=schema.GAMES_COLUMNS)\\n\\n\\ndef load_combine_tracking(\\n    *,\\n    players_only: bool = True,\\n    columns: Optional[List[str]] = None,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Load the 463k-row Combine tracking table (~80 MiB, fits in memory).\\"\\"\\"\\n    usecols = columns or schema.COMBINE_TRACKING_COLUMNS\\n    df = read_csv(\\n        paths.COMBINE_TRACKING,\\n        schema.COMBINE_TRACKING_DTYPES,\\n        usecols=usecols,\\n        validate_against=schema.COMBINE_TRACKING_COLUMNS,\\n    )\\n    if players_only and \\"entity_type\\" in df.columns:\\n        before = len(df)\\n        df = df[df[\\"entity_type\\"] == \\"PLAYER\\"].copy()\\n        log.info(\\"combine_tracking: kept %s/%s PLAYER frames\\", len(df), before)\\n    return df\\n\\n\\ndef csv_columns(name: str) -> List[str]:\\n    \\"\\"\\"Header of a source CSV without reading the body.\\"\\"\\"\\n    path = paths.DATA_DIR / name\\n    if not path.exists():\\n        raise FileNotFoundError(f\\"Missing competition file: {path}\\")\\n    return pd.read_csv(path, nrows=0).columns.tolist()\\n\\n\\ndef load_player_play(usecols: Optional[List[str]] = None) -> pd.DataFrame:\\n    \\"\\"\\"Load in-game Next Gen Stats player-play table (316k rows, 64 cols).\\"\\"\\"\\n    path = paths.DATA_DIR / paths.PLAYER_PLAY\\n    if not path.exists():\\n        raise FileNotFoundError(f\\"Missing competition file: {path}\\")\\n    # only the identity columns are contractual; the metric columns are many\\n    # and we select them by name, so extra columns must not raise a warning\\n    _validate_header(path, schema.PLAYER_PLAY_ID_COLUMNS, allow_extra=True)\\n    return pd.read_csv(path, usecols=usecols, low_memory=False)\\n\\n\\ndef iter_game_tracking(\\n    season: int,\\n    *,\\n    chunksize: int = 1_500_000,\\n    usecols: Optional[List[str]] = None,\\n) -> Iterable[pd.DataFrame]:\\n    \\"\\"\\"Stream one season of 10 Hz in-game tracking (up to ~4.3M rows).\\"\\"\\"\\n    name = paths.GAME_TRACKING[season]\\n    path = paths.DATA_DIR / name\\n    if not path.exists():\\n        raise FileNotFoundError(f\\"Missing competition file: {path}\\")\\n    _validate_header(path, schema.GAME_TRACKING_COLUMNS)\\n    return pd.read_csv(\\n        path,\\n        dtype=schema.GAME_TRACKING_DTYPES,\\n        usecols=usecols or schema.GAME_TRACKING_COLUMNS,\\n        chunksize=chunksize,\\n        low_memory=False,\\n    )\\n\\n\\ndef load_player_play_with_play_context(usecols: Optional[List[str]] = None) -> pd.DataFrame:\\n    \\"\\"\\"player_play enriched with games.csv season/week information.\\"\\"\\"\\n    pp = load_player_play(usecols=usecols)\\n    games = load_games()[[\\"game_id\\", \\"season\\", \\"season_type\\", \\"week\\"]]\\n    merged = pp.merge(games, on=\\"game_id\\", how=\\"left\\", validate=\\"many_to_one\\")\\n    return merged\\n\\n\\n# ---------------------------------------------------------------------------\\n# Reference / derived tables\\n# ---------------------------------------------------------------------------\\ndef position_group_table() -> pd.DataFrame:\\n    \\"\\"\\"players.csv plus a ``position_group`` column used to stratify analyses.\\"\\"\\"\\n    players = load_players().copy()\\n    players[\\"position_group\\"] = (\\n        players[\\"nfl_position\\"].map(schema.POSITION_GROUP).fillna(\\"OTHER\\")\\n    )\\n    return players\\n\\n\\ndef build_analysis_table(\\n    combine_player_features: pd.DataFrame,\\n    ingame_player_features: pd.DataFrame,\\n    *,\\n    force: bool = False,\\n) -> pd.DataFrame:\\n    \\"\\"\\"One row per prospect: identity + traditional combine + sensors + outcomes.\\"\\"\\"\\n\\n    def _build() -> pd.DataFrame:\\n        players = position_group_table()\\n        results = load_combine_results().drop(columns=[\\"draft_year\\"], errors=\\"ignore\\")\\n        career = load_career_successes()\\n\\n        players[\\"birth_date\\"] = pd.to_datetime(players[\\"birth_date\\"], errors=\\"coerce\\")\\n        players[\\"age_at_draft\\"] = (\\n            players[\\"draft_year\\"] - players[\\"birth_date\\"].dt.year\\n        ).astype(\\"Float64\\")\\n\\n        table = (\\n            players.merge(results, on=\\"nfl_id\\", how=\\"left\\", validate=\\"one_to_one\\")\\n            .merge(career, on=\\"nfl_id\\", how=\\"left\\", validate=\\"one_to_one\\")\\n            .merge(combine_player_features, on=\\"nfl_id\\", how=\\"left\\")\\n            .merge(ingame_player_features, on=\\"nfl_id\\", how=\\"left\\")\\n        )\\n\\n        # --- derived exposure / opportunity controls ---------------------\\n        table[\\"draft_capital\\"] = table[\\"draft_overall_pick\\"].fillna(300.0)\\n        table[\\"log_draft_capital\\"] = np.log1p(table[\\"draft_capital\\"])\\n        table[\\"is_udfa\\"] = table[\\"draft_overall_pick\\"].isna().astype(int)\\n\\n        table[\\"seasons_observed\\"] = (2025 - table[\\"draft_year\\"] + 1).astype(\\"Int64\\")\\n        for col in (\\n            \\"career_offensive_snaps\\",\\n            \\"career_defensive_snaps\\",\\n            \\"career_special_teams_snaps\\",\\n        ):\\n            per_season = f\\"{col}_per_season\\"\\n            table[per_season] = (\\n                table[col] / table[\\"seasons_observed\\"].astype(\\"float64\\")\\n            )\\n        table[\\"career_total_snaps\\"] = table[\\n            [\\n                \\"career_offensive_snaps\\",\\n                \\"career_defensive_snaps\\",\\n                \\"career_special_teams_snaps\\",\\n            ]\\n        ].sum(axis=1, min_count=1)\\n        table[\\"snaps_per_season\\"] = (\\n            table[\\"career_total_snaps\\"] / table[\\"seasons_observed\\"].astype(\\"float64\\")\\n        )\\n        table[\\"honors_score\\"] = (\\n            table[\\"ap_all_pro_1st_team\\"].fillna(0)\\n            + table[\\"ap_all_pro_2nd_team\\"].fillna(0)\\n            + table[\\"pro_bowl_original_ballot\\"].fillna(0)\\n        )\\n        return table\\n\\n    return cached(paths.CACHE_ANALYSIS_TABLE, _build, force=force)\\n", "reliability": "\\"\\"\\"Split-half reliability screening for sensor-derived traits.\\n\\nWhy this module exists\\n----------------------\\nA correlation between a sensor trait and an NFL outcome can only be as large as\\nthe trait\'s *reliability* allows. If a trait measured on half of an athlete\'s\\nreps does not predict the same trait measured on the other half, then the trait\\nis mostly measurement noise, and any correlation with an outcome is attenuated\\ntowards zero by a known factor:\\n\\n    r_observed  =  r_true * sqrt(reliability_trait * reliability_outcome)\\n\\nAt 10 Hz optical sampling, a change-of-direction \\"shape\\" metric such as\\n``speed_retention = exit_speed / entry_speed`` compounds two noisy speed\\nestimates inside a 0.2 s window. The competition asks for a metric that a team\\ncan act on, so reliability is not a technicality here -- it is the difference\\nbetween a scouting grade and a random number.\\n\\nScreening traits on reliability *before* looking at any outcome is also the\\ncheapest available defence against false discovery: it removes the noisiest\\ncandidates from the search space without ever consulting the labels.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom pathlib import Path\\nfrom typing import Dict, Iterable, List, Optional\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom . import paths\\n\\nlog = logging.getLogger(__name__)\\n\\nCACHE_NAME = \\"trait_reliability.parquet\\"\\nREPORT_NAME = \\"reliability_report.csv\\"\\n\\n# traits whose value is a level or a timing, versus those formed as a ratio or a\\n# difference of two noisy quantities. Used only for interpretation/reporting.\\nLEVEL_FEATURES = {\\n    \\"peak_speed\\", \\"mean_speed\\", \\"peak_accel\\", \\"max_abs_accel\\", \\"entry_speed\\",\\n    \\"exit_speed\\", \\"cut_min_speed\\", \\"turn_angle_deg\\", \\"turn_duration_s\\",\\n    \\"turn_rate_deg_s\\", \\"max_turn_rate_deg_s\\", \\"total_abs_turn_deg\\",\\n    \\"time_to_peak_speed\\", \\"duration_s\\", \\"path_length\\", \\"net_displacement\\",\\n}\\nRATIO_FEATURES = {\\n    \\"speed_retention\\", \\"speed_loss\\", \\"brake_rate\\", \\"exit_accel_rate\\",\\n    \\"regain_time_90\\", \\"regain_speed_slope\\", \\"path_efficiency\\",\\n    \\"cut_radius_proxy\\", \\"turn_start_frac\\",\\n}\\n\\n# The pool the reliability screen is computed over: every detected cut from\\n# every drill that actually contains a change of direction.\\nPOOL_EXCLUDED_DRILLS = {\\"FORTY_YARD_DASH\\"}\\n\\n\\ndef trait_kind(feature: str) -> str:\\n    if feature in RATIO_FEATURES:\\n        return \\"ratio/difference\\"\\n    if feature in LEVEL_FEATURES:\\n        return \\"level/timing\\"\\n    return \\"other\\"\\n\\n\\ndef split_half_reliability(\\n    attempts: pd.DataFrame,\\n    feature: str,\\n    *,\\n    n_rep: int = 120,\\n    min_players: int = 40,\\n    seed: int = 0,\\n) -> Dict[str, float]:\\n    \\"\\"\\"Reliability of a per-cut feature, by randomly splitting each player\'s cuts.\\n\\n    Important detail: the halves are formed from the **raw** feature values. A\\n    previous version centred the feature within player before splitting, which\\n    forces the two half-means to be negatively correlated by construction and\\n    produced reliabilities of about -5.8. The estimate here is the plain\\n    Spearman-Brown corrected correlation between half-means across players.\\n    \\"\\"\\"\\n    if feature not in attempts.columns:\\n        return {\\"reliability\\": np.nan, \\"n_cuts\\": 0, \\"n_players\\": 0}\\n\\n    pool = attempts[\\n        (pd.to_numeric(attempts.get(\\"n_turns\\"), errors=\\"coerce\\").fillna(0) > 0)\\n        & (~attempts[\\"drill_type\\"].isin(POOL_EXCLUDED_DRILLS))\\n    ]\\n    pool = pool[pd.to_numeric(pool[feature], errors=\\"coerce\\").notna()]\\n    if len(pool) < 200:\\n        return {\\"reliability\\": np.nan, \\"n_cuts\\": int(len(pool)), \\"n_players\\": 0}\\n\\n    rng = np.random.default_rng(seed)\\n    estimates: List[float] = []\\n    for _ in range(n_rep):\\n        shuffled = pool.sample(frac=1.0, random_state=int(rng.integers(1 << 31)))\\n        shuffled = shuffled.assign(_half=np.arange(len(shuffled)) % 2)\\n        a = shuffled[shuffled[\\"_half\\"] == 0].groupby(\\"nfl_id\\")[feature].mean()\\n        b = shuffled[shuffled[\\"_half\\"] == 1].groupby(\\"nfl_id\\")[feature].mean()\\n        joined = pd.concat([a, b], axis=1).dropna()\\n        if len(joined) < min_players:\\n            continue\\n        r = joined.corr().iloc[0, 1]\\n        if np.isfinite(r) and r > -0.95:\\n            estimates.append(2 * r / (1 + r))\\n    return {\\n        \\"reliability\\": float(np.mean(estimates)) if estimates else np.nan,\\n        \\"reliability_sd\\": float(np.std(estimates)) if estimates else np.nan,\\n        \\"n_cuts\\": int(len(pool)),\\n        \\"n_players\\": int(pool[\\"nfl_id\\"].nunique()),\\n        \\"n_rep\\": len(estimates),\\n    }\\n\\n\\ndef build_reliability_table(\\n    attempts: pd.DataFrame,\\n    features: Optional[Iterable[str]] = None,\\n    *,\\n    force: bool = False,\\n    n_rep: int = 120,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Reliability for every per-cut feature, cached as parquet + CSV.\\"\\"\\"\\n    cache = paths.CACHE_DIR / CACHE_NAME\\n    if cache.exists() and not force:\\n        cached = _read_parquet(cache)\\n        if cached is not None:\\n            return cached\\n\\n    if features is None:\\n        features = [\\n            c\\n            for c in attempts.select_dtypes(include=[np.number]).columns\\n            if c not in {\\"draft_year\\", \\"nfl_id\\", \\"attempt\\", \\"n_frames\\", \\"mean_dt\\"}\\n        ]\\n\\n    rows: List[Dict[str, object]] = []\\n    for feature in sorted(features):\\n        stats = split_half_reliability(attempts, feature, n_rep=n_rep)\\n        if not np.isfinite(stats.get(\\"reliability\\", np.nan)):\\n            continue\\n        rows.append({\\"feature\\": feature, \\"kind\\": trait_kind(feature), **stats})\\n        log.info(\\"reliability %-24s %.3f\\", feature, stats[\\"reliability\\"])\\n\\n    out = pd.DataFrame(rows).sort_values(\\"reliability\\", ascending=False)\\n    paths.CACHE_DIR.mkdir(parents=True, exist_ok=True)\\n    try:\\n        out.to_parquet(cache, index=False)\\n    except Exception as exc:  # noqa: BLE001\\n        log.warning(\\"parquet unavailable (%s); reliability cache written as CSV only\\", exc)\\n    # the CSV cache twin is what makes this module work without a parquet engine\\n    out.to_csv(cache.with_suffix(\\".csv\\"), index=False)\\n    out.to_csv(paths.REPORT_DIR / REPORT_NAME, index=False)\\n    return out\\n\\n\\ndef _read_parquet(path: Path) -> Optional[pd.DataFrame]:\\n    \\"\\"\\"Read a parquet cache, or return None when no engine is installed.\\n\\n    pyarrow is an optional dependency here: the CSV twin written beside every\\n    parquet cache is authoritative enough for reporting, so a missing engine\\n    must degrade rather than raise.\\n    \\"\\"\\"\\n    try:\\n        return pd.read_parquet(path)\\n    except Exception as exc:  # noqa: BLE001\\n        log.info(\\"parquet cache %s unreadable (%s); falling back to CSV\\", path.name, exc)\\n        return None\\n\\n\\ndef load_reliability_table() -> Optional[pd.DataFrame]:\\n    cache = paths.CACHE_DIR / CACHE_NAME\\n    if cache.exists():\\n        cached = _read_parquet(cache)\\n        if cached is not None:\\n            return cached\\n    twin = cache.with_suffix(\\".csv\\")\\n    if twin.exists():\\n        return pd.read_csv(twin)\\n    csv = paths.REPORT_DIR / REPORT_NAME\\n    if csv.exists():\\n        return pd.read_csv(csv)\\n    return None\\n\\n\\ndef reliability_lookup() -> Dict[str, float]:\\n    \\"\\"\\"``feature -> reliability`` mapping, empty when not yet computed.\\"\\"\\"\\n    table = load_reliability_table()\\n    if table is None or table.empty:\\n        return {}\\n    return dict(zip(table[\\"feature\\"], table[\\"reliability\\"]))\\n\\n\\ndef unreliability_ceiling(reliability: float) -> float:\\n    \\"\\"\\"Largest correlation the trait could show with a perfectly measured outcome.\\n\\n    This is the number that turns an apparent null result into an informative\\n    one: an observed r of 0.13 from a trait with reliability 0.11 implies a\\n    disattenuated r of about 0.39, so the honest statement is \\"underpowered\\",\\n    not \\"no relationship\\".\\n    \\"\\"\\"\\n    if not np.isfinite(reliability) or reliability <= 0:\\n        return 0.0\\n    return float(np.sqrt(reliability))\\n", "kinematics": "\\"\\"\\"Combine 10 Hz tracking -> interpretable movement kinematics.\\n\\nThis module is the analytical heart of the solution. It converts the raw\\nper-frame optical tracking of a Combine drill attempt into a small set of\\n*coach-readable* movement traits, with the explicit goal of separating the\\n\\"stopwatch\\" component of a drill (total time, which is already in\\n``combine_results.csv``) from the *shape* of the movement (how the athlete\\nbraked, how much speed survived the cut, and how quickly they re-accelerated).\\n\\nFeature families\\n----------------\\n``speed``       straight-line capability: peak / mean speed, acceleration.\\n``path``        geometry: path length, net displacement, path efficiency.\\n``turn``        change-of-direction: entry/exit speed, retention, angles.\\n``brake``       deceleration into a break.\\n``reaccel``     acceleration out of a break, and time to regain entry speed.\\n``asym``        left-vs-right asymmetry of the above.\\n\\nTwo design rules are enforced throughout:\\n\\n1. 10 Hz optical speed is noisy. Every differentiation happens on a\\n   Savitzky-Golay smoothed signal, never on the raw ``a`` column.\\n2. Every feature is defined per *attempt* (``event_id``). Player-level\\n   aggregation is a separate, explicit step so that \\"best attempt\\" and \\"mean\\n   attempt\\" can both be analysed.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom dataclasses import dataclass, field\\nfrom typing import Dict, List, Optional, Tuple\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nlog = logging.getLogger(__name__)\\n\\ntry:  # pragma: no cover - scipy is a declared dependency but stay defensive\\n    from scipy.signal import savgol_filter\\n\\n    _HAVE_SCIPY = True\\nexcept Exception:  # pragma: no cover\\n    _HAVE_SCIPY = False\\n\\n# ---------------------------------------------------------------------------\\n# Tunable parameters (single source of truth; surfaced in the writeup appendix)\\n# ---------------------------------------------------------------------------\\nSMOOTH_WINDOW_S = 0.30          # Savitzky-Golay window in seconds (3 frames @10Hz)\\nSMOOTH_POLYORDER = 2\\nTURN_RATE_THRESHOLD_DEG_S = 60.0  # angular velocity that counts as \\"a cut\\"\\nTURN_MIN_ANGLE_DEG = 25.0         # ignore micro-corrections below this total angle\\nTURN_MERGE_GAP_S = 0.20           # merge turn segments separated by < 0.2 s\\nSPEED_WINDOW_S = 0.20             # window used for entry / exit speed means\\nBRAKE_LOOKBACK_S = 0.50           # how far before the cut to look for braking\\nREACCEL_LOOKAHEAD_S = 0.60        # how far after the cut to look for re-acceleration\\nREGAIN_TARGET = 0.90              # \\"regained\\" threshold relative to entry speed\\nREGAIN_HORIZON_S = 2.00           # give up after this long\\n\\n\\n# ---------------------------------------------------------------------------\\n# Small numeric helpers\\n# ---------------------------------------------------------------------------\\ndef _smooth(values: np.ndarray, dt: float, window_s: float = SMOOTH_WINDOW_S) -> np.ndarray:\\n    \\"\\"\\"Savitzky-Golay smoothing with a hard fallback to a centred moving mean.\\"\\"\\"\\n    n = len(values)\\n    if n < 3:\\n        return values.astype(float)\\n    if _HAVE_SCIPY:\\n        fps = 1.0 / max(dt, 1e-3)\\n        win = int(round(window_s * fps))\\n        win = max(3, win)\\n        if win % 2 == 0:\\n            win += 1\\n        win = min(win, n if n % 2 == 1 else n - 1)\\n        if win >= 3:\\n            try:\\n                return savgol_filter(values.astype(float), win, SMOOTH_POLYORDER)\\n            except Exception:  # pragma: no cover\\n                pass\\n    k = max(1, int(round(window_s * 10 / 2)))\\n    return (\\n        pd.Series(values, dtype=\\"float64\\")\\n        .rolling(2 * k + 1, center=True, min_periods=1)\\n        .mean()\\n        .to_numpy()\\n    )\\n\\n\\ndef angular_velocity_deg_s(dir_deg: np.ndarray, t_sec: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Signed turn rate in deg/s, robust to the 0/360 wrap-around.\\n\\n    Returns an array of length ``len(dir_deg)`` aligned to the frame index\\n    (first element is back-filled from the second).\\n    \\"\\"\\"\\n    if len(dir_deg) < 2:\\n        return np.zeros(len(dir_deg))\\n    rad = np.deg2rad(np.asarray(dir_deg, dtype=float))\\n    delta = np.diff(rad)\\n    delta = (delta + np.pi) % (2.0 * np.pi) - np.pi  # wrap to [-pi, pi)\\n    dt = np.diff(t_sec)\\n    dt = np.where(dt <= 0, np.nan, dt)\\n    omega = np.degrees(delta) / dt\\n    omega = np.concatenate([[omega[0]], omega])\\n    return np.nan_to_num(omega, nan=0.0, posinf=0.0, neginf=0.0)\\n\\n\\ndef _segments_from_mask(mask: np.ndarray, t_sec: np.ndarray, gap_s: float) -> List[Tuple[int, int]]:\\n    \\"\\"\\"Contiguous True runs in ``mask``, merging runs closer than ``gap_s``.\\"\\"\\"\\n    idx = np.flatnonzero(mask)\\n    if idx.size == 0:\\n        return []\\n    splits = np.flatnonzero(np.diff(idx) > 1) + 1\\n    runs = np.split(idx, splits)\\n    merged: List[Tuple[int, int]] = []\\n    for run in runs:\\n        start, end = int(run[0]), int(run[-1])\\n        if merged and (t_sec[start] - t_sec[merged[-1][1]]) <= gap_s:\\n            merged[-1] = (merged[-1][0], end)\\n        else:\\n            merged.append((start, end))\\n    return merged\\n\\n\\ndef _window_mean(values: np.ndarray, t: np.ndarray, lo: float, hi: float) -> float:\\n    sel = (t >= lo) & (t <= hi)\\n    if not np.any(sel):\\n        return float(\\"nan\\")\\n    return float(np.nanmean(values[sel]))\\n\\n\\ndef _regain_time(\\n    speed: np.ndarray,\\n    t: np.ndarray,\\n    start_idx: int,\\n    entry_speed: float,\\n    horizon_s: float = REGAIN_HORIZON_S,\\n) -> float:\\n    \\"\\"\\"Seconds after ``start_idx`` until speed >= REGAIN_TARGET * entry_speed.\\n\\n    ``nan`` means the athlete never got back up to speed inside the horizon\\n    (i.e. a censored observation, not a missing one).\\n    \\"\\"\\"\\n    if not np.isfinite(entry_speed) or entry_speed <= 0:\\n        return float(\\"nan\\")\\n    target = REGAIN_TARGET * entry_speed\\n    t0 = t[start_idx]\\n    sel = (t >= t0) & (t <= t0 + horizon_s)\\n    if not np.any(sel):\\n        return float(\\"nan\\")\\n    hits = np.flatnonzero(sel & (speed >= target))\\n    if hits.size == 0:\\n        return float(\\"nan\\")\\n    return float(t[hits[0]] - t0)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Per-attempt feature extraction\\n# ---------------------------------------------------------------------------\\nATTEMPT_FEATURE_COLUMNS: List[str] = [\\n    # identity\\n    \\"draft_year\\", \\"event_id\\", \\"nfl_id\\", \\"drill_type\\", \\"drill_name\\", \\"attempt\\",\\n    # coverage\\n    \\"n_frames\\", \\"duration_s\\", \\"mean_dt\\",\\n    # speed\\n    \\"peak_speed\\", \\"mean_speed\\", \\"peak_accel\\", \\"peak_decel\\", \\"max_abs_accel\\",\\n    \\"time_to_peak_speed\\", \\"distance\\",\\n    # path geometry\\n    \\"path_length\\", \\"net_displacement\\", \\"path_efficiency\\",\\n    # turning\\n    \\"total_abs_turn_deg\\", \\"max_turn_rate_deg_s\\", \\"n_turns\\",\\n    # main cut\\n    \\"turn_angle_deg\\", \\"turn_duration_s\\", \\"turn_rate_deg_s\\",\\n    \\"entry_speed\\", \\"exit_speed\\", \\"cut_min_speed\\",\\n    \\"speed_retention\\", \\"speed_loss\\", \\"brake_rate\\", \\"exit_accel_rate\\",\\n    \\"regain_time_90\\", \\"regain_speed_slope\\", \\"cut_radius_proxy\\",\\n    # signed direction of the main cut (for asymmetry analysis)\\n    \\"turn_sign\\", \\"turn_start_frac\\",\\n]\\n\\n\\n@dataclass\\nclass AttemptKinematics:\\n    \\"\\"\\"Kinematic summary of a single drill attempt.\\"\\"\\"\\n\\n    meta: Dict[str, object] = field(default_factory=dict)\\n    features: Dict[str, float] = field(default_factory=dict)\\n    # smoothed series kept for figure generation / diagnostics\\n    series: Optional[Dict[str, np.ndarray]] = None\\n\\n\\ndef extract_attempt_kinematics(\\n    frames: pd.DataFrame,\\n    *,\\n    keep_series: bool = False,\\n) -> AttemptKinematics:\\n    \\"\\"\\"Compute kinematics for one attempt (one ``event_id`` / one player).\\"\\"\\"\\n    frames = frames.sort_values(\\"time\\")\\n    t_raw = frames[\\"time\\"].to_numpy(dtype=\\"datetime64[ns]\\")\\n    t = (t_raw - t_raw[0]).astype(\\"float64\\") / 1e9\\n\\n    # de-duplicate timestamps (optical systems occasionally emit ties)\\n    if len(t) > 1:\\n        keep = np.concatenate([[True], np.diff(t) > 0])\\n        t = t[keep]\\n        frames = frames.iloc[keep]\\n\\n    x = frames[\\"x\\"].to_numpy(dtype=float)\\n    y = frames[\\"y\\"].to_numpy(dtype=float)\\n    s_raw = frames[\\"s\\"].to_numpy(dtype=float)\\n    dir_deg = frames[\\"dir\\"].to_numpy(dtype=float)\\n    dis_raw = frames[\\"dis\\"].to_numpy(dtype=float)\\n\\n    meta = {\\n        \\"draft_year\\": frames[\\"draft_year\\"].iloc[0],\\n        \\"event_id\\": frames[\\"event_id\\"].iloc[0],\\n        \\"nfl_id\\": frames[\\"nfl_id\\"].iloc[0],\\n        \\"drill_type\\": frames[\\"drill_type\\"].iloc[0],\\n        \\"drill_name\\": frames[\\"drill_name\\"].iloc[0],\\n        \\"attempt\\": frames[\\"attempt\\"].iloc[0],\\n    }\\n\\n    n = len(t)\\n    dt = float(np.median(np.diff(t))) if n > 1 else 0.1\\n\\n    feats: Dict[str, float] = {\\"n_frames\\": n, \\"mean_dt\\": dt}\\n    feats[\\"duration_s\\"] = float(t[-1] - t[0]) if n > 1 else 0.0\\n\\n    if n < 5:\\n        for col in ATTEMPT_FEATURE_COLUMNS:\\n            feats.setdefault(col, np.nan)\\n        return AttemptKinematics(meta=meta, features=feats)\\n\\n    # ---- smoothing -------------------------------------------------------\\n    s = _smooth(s_raw, dt)\\n    seconds = np.clip(s, 0, None)\\n    # acceleration from the smoothed speed signal (never from the raw `a`)\\n    accel = np.gradient(seconds, t, edge_order=1)\\n\\n    feats[\\"peak_speed\\"] = float(np.nanmax(seconds))\\n    feats[\\"mean_speed\\"] = float(np.nanmean(seconds))\\n    feats[\\"peak_accel\\"] = float(np.nanmax(accel))\\n    feats[\\"peak_decel\\"] = float(np.nanmin(accel))\\n    feats[\\"max_abs_accel\\"] = float(np.nanmax(np.abs(accel)))\\n    feats[\\"time_to_peak_speed\\"] = float(t[int(np.nanargmax(seconds))])\\n\\n    # ---- path geometry ---------------------------------------------------\\n    step = np.hypot(np.diff(x), np.diff(y))\\n    path_length = float(np.nansum(step))\\n    net = float(np.hypot(x[-1] - x[0], y[-1] - y[0]))\\n    feats[\\"path_length\\"] = path_length\\n    feats[\\"net_displacement\\"] = net\\n    # prefer the tracked `dis` sum when it is plausible, else the chord sum\\n    dis_sum = float(np.nansum(dis_raw))\\n    feats[\\"distance\\"] = dis_sum if dis_sum > 0 else path_length\\n    feats[\\"path_efficiency\\"] = net / path_length if path_length > 0 else np.nan\\n\\n    # ---- turning ---------------------------------------------------------\\n    omega = angular_velocity_deg_s(dir_deg, t)\\n    omega_s = _smooth(omega, dt, window_s=0.2)\\n    feats[\\"total_abs_turn_deg\\"] = float(np.nansum(np.abs(omega_s) * dt))\\n    feats[\\"max_turn_rate_deg_s\\"] = float(np.nanmax(np.abs(omega_s)))\\n\\n    mask = np.abs(omega_s) > TURN_RATE_THRESHOLD_DEG_S\\n    runs = _segments_from_mask(mask, t, TURN_MERGE_GAP_S)\\n\\n    turns: List[Dict[str, float]] = []\\n    for start, end in runs:\\n        # uniform 10 Hz sampling: total signed rotation = mean rate * elapsed\\n        signed_angle = float(np.nansum(omega_s[start : end + 1]) * dt)\\n        angle = abs(signed_angle)\\n        if angle < TURN_MIN_ANGLE_DEG:\\n            continue\\n        t_start, t_end = t[start], t[end]\\n        entry = _window_mean(seconds, t, t_start - SPEED_WINDOW_S, t_start)\\n        exit_ = _window_mean(seconds, t, t_end, t_end + SPEED_WINDOW_S)\\n        seg = slice(start, end + 1)\\n        cut_min = float(np.nanmin(seconds[seg]))\\n        brake = float(np.nanmin(accel[(t >= t_start - BRAKE_LOOKBACK_S) & (t <= t_start)]))\\n        reacc = float(np.nanmax(accel[(t >= t_end) & (t <= t_end + REACCEL_LOOKAHEAD_S)]))\\n        retention = exit_ / entry if (np.isfinite(entry) and entry > 0) else np.nan\\n        turns.append(\\n            {\\n                \\"signed_angle\\": signed_angle,\\n                \\"angle\\": angle,\\n                \\"start_idx\\": start,\\n                \\"t_start\\": t_start,\\n                \\"t_end\\": t_end,\\n                \\"duration\\": max(t_end - t_start, dt),\\n                \\"entry_speed\\": entry,\\n                \\"exit_speed\\": exit_,\\n                \\"cut_min_speed\\": cut_min,\\n                \\"speed_retention\\": retention,\\n                \\"speed_loss\\": entry - exit_ if np.isfinite(entry) and np.isfinite(exit_) else np.nan,\\n                \\"brake_rate\\": brake,\\n                \\"exit_accel_rate\\": reacc,\\n                \\"regain_time_90\\": _regain_time(seconds, t, end, entry),\\n                \\"turn_rate\\": angle / max(t_end - t_start, dt),\\n            }\\n        )\\n\\n    feats[\\"n_turns\\"] = len(turns)\\n    if turns:\\n        main = max(turns, key=lambda d: d[\\"angle\\"])\\n        feats[\\"turn_angle_deg\\"] = main[\\"angle\\"]\\n        feats[\\"turn_sign\\"] = float(np.sign(main[\\"signed_angle\\"]))\\n        feats[\\"turn_duration_s\\"] = main[\\"duration\\"]\\n        feats[\\"turn_rate_deg_s\\"] = main[\\"turn_rate\\"]\\n        feats[\\"entry_speed\\"] = main[\\"entry_speed\\"]\\n        feats[\\"exit_speed\\"] = main[\\"exit_speed\\"]\\n        feats[\\"cut_min_speed\\"] = main[\\"cut_min_speed\\"]\\n        feats[\\"speed_retention\\"] = main[\\"speed_retention\\"]\\n        feats[\\"speed_loss\\"] = main[\\"speed_loss\\"]\\n        feats[\\"brake_rate\\"] = main[\\"brake_rate\\"]\\n        feats[\\"exit_accel_rate\\"] = main[\\"exit_accel_rate\\"]\\n        feats[\\"regain_time_90\\"] = main[\\"regain_time_90\\"]\\n        feats[\\"turn_start_frac\\"] = main[\\"t_start\\"] / feats[\\"duration_s\\"] if feats[\\"duration_s\\"] else np.nan\\n        # how fast they climb back out of the cut\'s speed minimum\\n        post = (t >= main[\\"t_end\\"]) & (t <= main[\\"t_end\\"] + REACCEL_LOOKAHEAD_S)\\n        if np.count_nonzero(post) >= 2:\\n            tt = t[post]\\n            ss = seconds[post]\\n            slope = np.polyfit(tt - tt[0], ss, 1)[0] if len(tt) > 1 else np.nan\\n        else:\\n            slope = np.nan\\n        feats[\\"regain_speed_slope\\"] = float(slope) if slope is not None else np.nan\\n        feats[\\"cut_radius_proxy\\"] = (\\n            main[\\"entry_speed\\"] / np.deg2rad(main[\\"turn_rate\\"])\\n            if main[\\"turn_rate\\"] > 0\\n            else np.nan\\n        )\\n    else:\\n        for col in (\\n            \\"turn_angle_deg\\", \\"turn_sign\\", \\"turn_duration_s\\", \\"turn_rate_deg_s\\",\\n            \\"entry_speed\\", \\"exit_speed\\", \\"cut_min_speed\\", \\"speed_retention\\",\\n            \\"speed_loss\\", \\"brake_rate\\", \\"exit_accel_rate\\", \\"regain_time_90\\",\\n            \\"turn_start_frac\\", \\"regain_speed_slope\\", \\"cut_radius_proxy\\",\\n        ):\\n            feats[col] = np.nan\\n\\n    series = None\\n    if keep_series:\\n        series = {\\"t\\": t, \\"speed\\": seconds, \\"accel\\": accel, \\"omega\\": omega_s, \\"x\\": x, \\"y\\": y}\\n    return AttemptKinematics(meta=meta, features=feats, series=series)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Table level driver\\n# ---------------------------------------------------------------------------\\ndef extract_all_attempt_features(\\n    tracking: pd.DataFrame,\\n    *,\\n    min_frames: int = 10,\\n    progress_every: int = 1000,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Apply :func:`extract_attempt_kinematics` to every attempt in the table.\\"\\"\\"\\n    tracking = tracking.dropna(subset=[\\"nfl_id\\"])\\n    keys = [\\"draft_year\\", \\"event_id\\", \\"nfl_id\\"]\\n    grouped = tracking.groupby(keys, sort=False, observed=True)\\n\\n    rows: List[Dict[str, object]] = []\\n    for i, (_, frames) in enumerate(grouped):\\n        if len(frames) < min_frames:\\n            continue\\n        kin = extract_attempt_kinematics(frames)\\n        rows.append({**kin.meta, **kin.features})\\n        if progress_every and (i + 1) % progress_every == 0:\\n            log.info(\\"kinematics: %s attempts processed\\", i + 1)\\n\\n    out = pd.DataFrame(rows)\\n    log.info(\\"extracted kinematics for %s attempts\\", len(out))\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------\\n# Player-level aggregation\\n# ---------------------------------------------------------------------------\\ndef aggregate_attempts_to_player(\\n    attempts: pd.DataFrame,\\n    *,\\n    agg: Tuple[str, ...] = (\\"mean\\", \\"max\\", \\"min\\"),\\n) -> pd.DataFrame:\\n    \\"\\"\\"Collapse attempt-level kinematics to one row per (nfl_id, drill_type).\\n\\n    Deliberately keeps ``best_*`` (max) and ``worst_*`` (min) alongside the mean\\n    because Combine athletes get several attempts and the coaching question is\\n    usually \\"what is this player capable of at his best?\\".\\n    \\"\\"\\"\\n    numeric = attempts.select_dtypes(include=[np.number]).columns.tolist()\\n    # identity / bookkeeping columns are NOT movement traits. Including them\\n    # would let \\"which attempt number this was\\" masquerade as a kinematic signal.\\n    drop = {\\"draft_year\\", \\"nfl_id\\", \\"attempt\\"}\\n    numeric = [c for c in numeric if c not in drop]\\n\\n    grouped = attempts.groupby([\\"nfl_id\\", \\"drill_type\\"], observed=True)[numeric]\\n    wide = grouped.agg(list(agg))\\n\\n    if isinstance(wide.columns, pd.MultiIndex):\\n        wide.columns = [f\\"{drill}_{stat}\\" for drill, stat in wide.columns]\\n    else:  # pragma: no cover - defensive\\n        wide.columns = [f\\"{c}_{agg[0]}\\" for c in wide.columns]\\n\\n    wide = wide.reset_index()\\n    wide[\\"n_attempts\\"] = (\\n        attempts.groupby([\\"nfl_id\\", \\"drill_type\\"], observed=True)\\n        .size()\\n        .reset_index(drop=True)\\n        .to_numpy()\\n    )\\n    wide[\\"drill_type\\"] = wide[\\"drill_type\\"].astype(str)\\n    wide[\\"drill_slug\\"] = (\\n        wide[\\"drill_type\\"].str.replace(\\"SKILL_DRILLS_\\", \\"SKILL_\\", regex=False).str.lower()\\n    )\\n    return wide\\n\\n\\ndef pivot_player_features(player_drill: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Wide table, one row per player, columns ``<drill>__<feature>__<stat>``.\\"\\"\\"\\n    value_cols = [\\n        c\\n        for c in player_drill.columns\\n        if c not in {\\"nfl_id\\", \\"drill_type\\", \\"drill_slug\\", \\"n_attempts\\"}\\n    ]\\n    melted = player_drill.melt(\\n        id_vars=[\\"nfl_id\\", \\"drill_slug\\"],\\n        value_vars=value_cols,\\n        var_name=\\"feature\\",\\n        value_name=\\"value\\",\\n    )\\n    melted[\\"column\\"] = (\\n        melted[\\"drill_slug\\"].astype(str) + \\"__\\" + melted[\\"feature\\"].astype(str)\\n    )\\n    wide = melted.pivot_table(\\n        index=\\"nfl_id\\", columns=\\"column\\", values=\\"value\\", aggfunc=\\"first\\"\\n    )\\n    wide.columns.name = None\\n    return wide.reset_index()\\n\\n\\n# ---------------------------------------------------------------------------\\n# Pooled, player-level change-of-direction traits\\n# ---------------------------------------------------------------------------\\n# Rationale (this is the analytical pivot of the whole solution):\\n# drill-specific COD traits are only computable for the players who ran *that*\\n# drill, and on this dataset 47-80% of prospects opt out of the 3-cone and\\n# short shuttle. That fragments a 510-player cohort into cells of n = 26-47 and\\n# destroys statistical power for exactly the traits the competition is about.\\n#\\n# A change-of-direction trait is a property of the *athlete*, not of the drill.\\n# Pooling every detected cut across every drill that actually contains a turn\\n# restores coverage to 510/510 (median 9 cuts per player) while remaining\\n# conceptually cleaner: the question becomes \\"how does this player move through\\n# cuts\\", which is what a position coach asks.\\n#\\n# The 40-yard dash is excluded from the pool: it is a straight line, and cuts\\n# \\"detected\\" in it are tracking noise (see schema.COD_ONLY_FEATURES).\\n\\nCOD_POOL_EXCLUDED_DRILLS = {\\"FORTY_YARD_DASH\\"}\\n\\nPOOLED_TRAIT_COLUMNS: List[str] = [\\n    \\"pooled_n_cuts\\",\\n    \\"pooled_n_drills\\",\\n    \\"pooled_turn_angle_mean\\",\\n    \\"pooled_entry_speed_mean\\",\\n    \\"pooled_exit_speed_mean\\",\\n    \\"pooled_retention_mean\\",\\n    \\"pooled_retention_best\\",\\n    \\"pooled_retention_worst\\",\\n    \\"pooled_retention_angle_adj\\",\\n    \\"pooled_speed_loss_mean\\",\\n    \\"pooled_brake_mean\\",\\n    \\"pooled_exit_accel_mean\\",\\n    \\"pooled_regain_time_mean\\",\\n    \\"pooled_regain_censored_rate\\",\\n    \\"pooled_cut_radius_mean\\",\\n    \\"pooled_retention_asym\\",\\n]\\n\\n\\ndef pooled_cod_player_features(\\n    attempts: pd.DataFrame,\\n    *,\\n    horizon_s: float = REGAIN_HORIZON_S,\\n) -> pd.DataFrame:\\n    \\"\\"\\"One row per player: change-of-direction traits pooled across all drills.\\n\\n    Two corrections are applied that matter for validity:\\n\\n    * **Angle adjustment.** Speed retention falls as the cut gets sharper, so a\\n      raw pooled mean partly measures which drills a player happened to run.\\n      ``pooled_retention_angle_adj`` is the residual of retention on turn angle,\\n      fitted across the pool, so that a player is compared to a player who cut\\n      by the same amount.\\n    * **Censoring.** ``regain_time_90`` is ``NaN`` when the athlete never got\\n      back to 90% of entry speed inside the horizon. That is a censored\\n      observation, not a missing one, so it is imputed at the horizon and the\\n      censoring rate is carried as its own feature.\\n    \\"\\"\\"\\n    if attempts.empty:\\n        return pd.DataFrame(columns=[\\"nfl_id\\"] + POOLED_TRAIT_COLUMNS)\\n\\n    cod = attempts[\\n        (pd.to_numeric(attempts.get(\\"n_turns\\"), errors=\\"coerce\\").fillna(0) > 0)\\n        & (~attempts[\\"drill_type\\"].isin(COD_POOL_EXCLUDED_DRILLS))\\n        & attempts[\\"speed_retention\\"].notna()\\n    ].copy()\\n\\n    if cod.empty:\\n        return pd.DataFrame(columns=[\\"nfl_id\\"] + POOLED_TRAIT_COLUMNS)\\n\\n    # angle-adjusted retention: remove the mechanical angle dependence\\n    ang = pd.to_numeric(cod[\\"turn_angle_deg\\"], errors=\\"coerce\\")\\n    ret = pd.to_numeric(cod[\\"speed_retention\\"], errors=\\"coerce\\")\\n    ok = ang.notna() & ret.notna()\\n    if ok.sum() > 30 and ang[ok].std() > 0:\\n        slope, intercept = np.polyfit(ang[ok], ret[ok], 1)\\n        cod[\\"retention_angle_adj\\"] = ret - (intercept + slope * ang)\\n    else:\\n        cod[\\"retention_angle_adj\\"] = ret - ret.mean()\\n\\n    # censored regain time -> impute at the horizon\\n    regain = pd.to_numeric(cod[\\"regain_time_90\\"], errors=\\"coerce\\")\\n    cod[\\"regain_censored\\"] = regain.isna().astype(float)\\n    cod[\\"regain_filled\\"] = regain.fillna(horizon_s)\\n\\n    # signed asymmetry: does the athlete retain more speed cutting one way?\\n    if \\"turn_sign\\" in cod.columns:\\n        left = (\\n            cod[cod[\\"turn_sign\\"] > 0].groupby(\\"nfl_id\\")[\\"speed_retention\\"].mean()\\n        )\\n        right = (\\n            cod[cod[\\"turn_sign\\"] < 0].groupby(\\"nfl_id\\")[\\"speed_retention\\"].mean()\\n        )\\n        asym = (left - right).rename(\\"pooled_retention_asym\\")\\n    else:  # pragma: no cover\\n        asym = pd.Series(dtype=float, name=\\"pooled_retention_asym\\")\\n\\n    g = cod.groupby(\\"nfl_id\\", observed=True)\\n    out = pd.DataFrame(\\n        {\\n            \\"pooled_n_cuts\\": g.size(),\\n            \\"pooled_n_drills\\": g[\\"drill_type\\"].nunique(),\\n            \\"pooled_turn_angle_mean\\": g[\\"turn_angle_deg\\"].mean(),\\n            \\"pooled_entry_speed_mean\\": g[\\"entry_speed\\"].mean(),\\n            \\"pooled_exit_speed_mean\\": g[\\"exit_speed\\"].mean(),\\n            \\"pooled_retention_mean\\": g[\\"speed_retention\\"].mean(),\\n            \\"pooled_retention_best\\": g[\\"speed_retention\\"].max(),\\n            \\"pooled_retention_worst\\": g[\\"speed_retention\\"].min(),\\n            \\"pooled_retention_angle_adj\\": g[\\"retention_angle_adj\\"].mean(),\\n            \\"pooled_speed_loss_mean\\": g[\\"speed_loss\\"].mean(),\\n            \\"pooled_brake_mean\\": g[\\"brake_rate\\"].mean(),\\n            \\"pooled_exit_accel_mean\\": g[\\"exit_accel_rate\\"].mean(),\\n            \\"pooled_regain_time_mean\\": g[\\"regain_filled\\"].mean(),\\n            \\"pooled_regain_censored_rate\\": g[\\"regain_censored\\"].mean(),\\n            \\"pooled_cut_radius_mean\\": g[\\"cut_radius_proxy\\"].mean(),\\n        }\\n    ).reset_index()\\n    out = out.merge(asym.reset_index(), on=\\"nfl_id\\", how=\\"left\\")\\n    return out\\n\\n\\ndef player_level_combine_features(attempts: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Public entry point: attempt table -> per-player wide feature table.\\"\\"\\"\\n    per_drill = aggregate_attempts_to_player(attempts)\\n    wide = pivot_player_features(per_drill)\\n    # position-agnostic \\"best across all drills\\" summaries\\n    overall = (\\n        attempts.groupby(\\"nfl_id\\", observed=True)\\n        .agg(\\n            all_drills_peak_speed=(\\"peak_speed\\", \\"max\\"),\\n            all_drills_mean_speed=(\\"mean_speed\\", \\"mean\\"),\\n            all_drills_peak_accel=(\\"peak_accel\\", \\"max\\"),\\n            all_drills_best_retention=(\\"speed_retention\\", \\"max\\"),\\n            all_drills_mean_retention=(\\"speed_retention\\", \\"mean\\"),\\n            all_drills_min_regain=(\\"regain_time_90\\", \\"min\\"),\\n            all_drills_turns=(\\"n_turns\\", \\"sum\\"),\\n            all_drills_path_eff=(\\"path_efficiency\\", \\"mean\\"),\\n        )\\n        .reset_index()\\n    )\\n    merged = wide.merge(overall, on=\\"nfl_id\\", how=\\"outer\\")\\n    pooled = pooled_cod_player_features(attempts)\\n    return merged.merge(pooled, on=\\"nfl_id\\", how=\\"outer\\")\\n\\n\\n# ---------------------------------------------------------------------------\\n# Validation against the official electronic timings\\n# ---------------------------------------------------------------------------\\ndef validate_against_official_timings(\\n    attempts: pd.DataFrame,\\n    combine_results: pd.DataFrame,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Sanity-check sensor durations against the stopwatch numbers.\\n\\n    This is the credibility figure for the whole analysis: if sensor-derived\\n    drill duration does not track the official electronic time, no downstream\\n    sensor feature can be trusted.\\n    \\"\\"\\"\\n    res = combine_results[[\\"nfl_id\\", \\"forty\\", \\"three_cone\\", \\"short_shuttle\\", \\"ten_yd_split\\"]]\\n    checks = [\\n        (\\"THREE_CONE_DRILL\\", \\"three_cone\\"),\\n        (\\"SHORT_SHUTTLE\\", \\"short_shuttle\\"),\\n        (\\"FORTY_YARD_DASH\\", \\"forty\\"),\\n    ]\\n    out: List[pd.DataFrame] = []\\n    for drill, official in checks:\\n        sub = attempts[attempts[\\"drill_type\\"] == drill][\\n            [\\"nfl_id\\", \\"attempt\\", \\"duration_s\\", \\"peak_speed\\"]\\n        ].merge(res, on=\\"nfl_id\\", how=\\"inner\\")\\n        sub = sub.dropna(subset=[\\"duration_s\\", official])\\n        if sub.empty:\\n            continue\\n        # attempts can include partial captures; keep the attempt whose duration\\n        # is closest to the official time (that is the one that was timed)\\n        sub[\\"abs_err\\"] = (sub[\\"duration_s\\"] - sub[official]).abs()\\n        best = sub.sort_values(\\"abs_err\\").groupby(\\"nfl_id\\", as_index=False).first()\\n        best[\\"drill_type\\"] = drill\\n        best[\\"official_metric\\"] = official\\n        out.append(best)\\n    if not out:\\n        return pd.DataFrame()\\n    return pd.concat(out, ignore_index=True)\\n", "ingame": "\\"\\"\\"In-game outcomes: what the prospect actually did on an NFL field.\\n\\nTwo complementary sources are used, and the fact that they are *independent*\\nis what makes the linkage analysis credible:\\n\\n``player_play.csv``\\n    Vendor/Next Gen Stats derived per-play metrics (separation, cushion,\\n    pass-rush get-off, time to pressure, EPA, ...). Rich but model-derived.\\n\\n``game_tracking_YYYY.csv``\\n    The raw 10 Hz optical tracking. We re-derive movement quantities ourselves\\n    (peak speed, distance, burst immediately after the snap) so the headline\\n    claim can be checked without trusting the vendor columns.\\n\\nOutcome naming convention\\n-------------------------\\n``<metric>_mean``, ``_p75``, ``_p25``, ``_rate`` etc. Every aggregate is\\ncomputed *within position group* only at analysis time; here we simply produce\\none row per ``nfl_id``.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom typing import Dict, List, Optional, Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom . import io, paths, schema\\n\\nlog = logging.getLogger(__name__)\\n\\n# ---------------------------------------------------------------------------\\n# player_play.csv -> per-player outcome table\\n# ---------------------------------------------------------------------------\\n# metric -> (source column, aggregation recipe)\\n_MEAN_METRICS = {\\n    \\"sep\\": \\"separation_at_pass_forward\\",\\n    \\"cushion\\": \\"cushion\\",\\n    \\"get_off\\": \\"player_get_off\\",\\n    \\"time_to_pressure\\": \\"time_to_pressure\\",\\n    \\"time_to_qb_hurry\\": \\"time_to_qb_hurry\\",\\n    \\"peak_pressure_prob_allowed\\": \\"peak_pressure_probability_allowed\\",\\n    \\"ttp_allowed\\": \\"time_to_pressure_allowed\\",\\n    \\"dropback_duration\\": \\"dropback_duration\\",\\n    \\"rush_encountered\\": \\"pass_rushers_encountered\\",\\n    \\"yac\\": \\"yards_after_catch\\",\\n    \\"eyac\\": \\"expected_yards_after_catch\\",\\n    \\"pass_length\\": \\"pass_length\\",\\n    \\"epa\\": \\"expected_points_added\\",\\n    \\"rec_yards\\": \\"rec_yards\\",\\n}\\n\\n_RATE_METRICS = {\\n    \\"target\\": \\"target\\",\\n    \\"sack\\": \\"sack\\",\\n    \\"tackle\\": \\"tackle\\",\\n    \\"assist\\": \\"assist\\",\\n    \\"tfl\\": \\"tackle_for_loss\\",\\n    \\"ff\\": \\"caused_forced_fumble\\",\\n    \\"fumble_rec\\": \\"recovered_fumble\\",\\n    \\"pressure_allowed\\": \\"pressure_allowed\\",\\n    \\"sack_allowed\\": \\"sack_allowed\\",\\n    \\"blitz\\": \\"blitzing\\",\\n    \\"quick_pressure\\": \\"quick_pressure\\",\\n}\\n\\nPLAYER_PLAY_USECOLS: List[str] = (\\n    [\\"game_id\\", \\"play_id\\", \\"nfl_id\\", \\"lined_up_position\\"]\\n    + sorted(set(_MEAN_METRICS.values()) | set(_RATE_METRICS.values()))\\n)\\n\\n\\ndef _to_numeric(series: pd.Series) -> pd.Series:\\n    if series.dtype == bool:\\n        return series.astype(\\"float64\\")\\n    return pd.to_numeric(series, errors=\\"coerce\\")\\n\\n\\ndef build_player_play_outcomes(\\n    *,\\n    min_plays: int = 20,\\n    usecols: Optional[Sequence[str]] = None,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Aggregate the 316k-row player-play table to one row per prospect.\\n\\n    ``min_plays`` is deliberately *not* used to drop rows -- players with very\\n    few snaps carry real information (they are the draft busts) -- but the play\\n    count is returned so the analysis can weight or filter explicitly.\\n    \\"\\"\\"\\n    cols = list(usecols) if usecols else PLAYER_PLAY_USECOLS\\n    # keep only columns that actually exist in this release of the file, so the\\n    # pipeline degrades gracefully rather than crashing on a schema tweak\\n    available = set(io.csv_columns(paths.PLAYER_PLAY))\\n    cols = [c for c in cols if c in available]\\n    missing_requested = sorted(set(PLAYER_PLAY_USECOLS) - available)\\n    if missing_requested:\\n        log.warning(\\"player_play is missing expected metric columns: %s\\", missing_requested)\\n    pp = io.load_player_play(usecols=cols)\\n    games = io.load_games()[[\\"game_id\\", \\"season\\", \\"season_type\\", \\"week\\"]]\\n    pp = pp.merge(games, on=\\"game_id\\", how=\\"left\\")\\n\\n    for col in _MEAN_METRICS.values():\\n        if col in pp.columns:\\n            pp[col] = _to_numeric(pp[col])\\n    for col in _RATE_METRICS.values():\\n        if col in pp.columns:\\n            pp[col] = _to_numeric(pp[col])\\n\\n    g = pp.groupby(\\"nfl_id\\", observed=True)\\n\\n    out = pd.DataFrame(index=g.size().index)\\n    out[\\"pp_plays\\"] = g.size()\\n    out[\\"pp_seasons\\"] = g[\\"season\\"].nunique()\\n    out[\\"pp_games\\"] = g[\\"game_id\\"].nunique()\\n\\n    # ---- central tendency + upper/lower tail for each continuous metric ----\\n    for short, col in _MEAN_METRICS.items():\\n        if col not in pp.columns:\\n            continue\\n        series = pp[col]\\n        out[f\\"{short}_mean\\"] = g[col].mean()\\n        out[f\\"{short}_n\\"] = g[col].count()\\n        out[f\\"{short}_p75\\"] = g[col].quantile(0.75)\\n        out[f\\"{short}_p25\\"] = g[col].quantile(0.25)\\n        out[f\\"{short}_max\\"] = g[col].max()\\n        # opportunities: fraction of the player\'s snaps on which the metric exists\\n        out[f\\"{short}_avail_rate\\"] = g[col].apply(lambda s: float(s.notna().mean()))\\n\\n    # ---- rate metrics ------------------------------------------------------\\n    for short, col in _RATE_METRICS.items():\\n        if col not in pp.columns:\\n            continue\\n        s = pp[col]\\n        out[f\\"{short}_rate\\"] = g[col].mean()\\n        out[f\\"{short}_per100\\"] = g[col].mean() * 100.0\\n\\n    # ---- composite outcomes that matter to NFL evaluators -----------------\\n    if {\\"expected_points_added\\"}.issubset(pp.columns):\\n        out[\\"epa_per_play\\"] = out[\\"epa_mean\\"]\\n    if {\\"separation_at_pass_forward\\", \\"target\\"}.issubset(pp.columns):\\n        # receiving yards per target = the canonical efficiency outcome\\n        ry = pp[\\"rec_yards\\"] if \\"rec_yards\\" in pp.columns else None\\n        if ry is not None:\\n            tg = pp[pp[\\"target\\"] == 1].groupby(\\"nfl_id\\", observed=True)\\n            out[\\"rec_yards_per_target\\"] = (\\n                tg[\\"rec_yards\\"].sum() / tg.size().replace(0, np.nan)\\n            )\\n            out[\\"yac_per_reception\\"] = (\\n                tg[\\"yards_after_catch\\"].sum() / tg.size().replace(0, np.nan)\\n            )\\n\\n    out = out.reset_index()\\n    out.attrs[\\"min_plays_hint\\"] = min_plays\\n    return out\\n\\n\\n# ---------------------------------------------------------------------------\\n# game_tracking_YYYY.csv -> raw-motion per-player features\\n# ---------------------------------------------------------------------------\\n_GAME_TRACKING_USECOLS = [\\n    \\"game_id\\", \\"play_id\\", \\"nfl_id\\", \\"time\\", \\"s\\", \\"a\\", \\"dis\\", \\"event\\",\\n]\\n# window (seconds) after the snap used for the \\"first-step burst\\" proxy\\nSNAP_BURST_WINDOW_S = 1.5\\nPLAY_FEATURE_COLUMNS = [\\n    \\"peak_speed\\", \\"mean_speed\\", \\"peak_accel\\", \\"peak_decel\\",\\n    \\"distance\\", \\"high_speed_distance\\", \\"n_frames\\",\\n    \\"snap_first_frame_s\\", \\"snap_speed_0p5\\", \\"snap_burst_peak_accel\\",\\n    \\"snap_burst_peak_speed\\", \\"snap_burst_distance\\", \\"snap_burst_n\\",\\n]\\n\\n\\ndef _snap_times_for(chunk: pd.DataFrame) -> pd.DataFrame:\\n    \\"\\"\\"Earliest ``ball_snap`` timestamp per (game_id, play_id).\\"\\"\\"\\n    snaps = chunk[chunk[\\"event\\"] == \\"ball_snap\\"][[\\"game_id\\", \\"play_id\\", \\"time\\"]]\\n    if snaps.empty:\\n        return pd.DataFrame(columns=[\\"game_id\\", \\"play_id\\", \\"snap_time\\"])\\n    return (\\n        snaps.groupby([\\"game_id\\", \\"play_id\\"], observed=True)[\\"time\\"]\\n        .min()\\n        .reset_index()\\n        .rename(columns={\\"time\\": \\"snap_time\\"})\\n    )\\n\\n\\ndef _play_features_for_chunk(\\n    chunk: pd.DataFrame, snap_times: pd.DataFrame\\n) -> pd.DataFrame:\\n    keys = [\\"game_id\\", \\"play_id\\", \\"nfl_id\\"]\\n    chunk = chunk.copy()\\n    chunk[\\"dis\\"] = pd.to_numeric(chunk[\\"dis\\"], errors=\\"coerce\\").fillna(0.0)\\n    chunk[\\"s\\"] = pd.to_numeric(chunk[\\"s\\"], errors=\\"coerce\\")\\n    chunk[\\"a\\"] = pd.to_numeric(chunk[\\"a\\"], errors=\\"coerce\\")\\n    chunk[\\"high_speed_dis\\"] = chunk[\\"dis\\"].where(chunk[\\"s\\"] >= 5.0, 0.0)\\n\\n    g = chunk.groupby(keys, sort=False, observed=True)\\n    feats = pd.DataFrame(\\n        {\\n            \\"peak_speed\\": g[\\"s\\"].max(),\\n            \\"mean_speed\\": g[\\"s\\"].mean(),\\n            \\"peak_accel\\": g[\\"a\\"].max(),\\n            \\"peak_decel\\": g[\\"a\\"].min(),\\n            \\"distance\\": g[\\"dis\\"].sum(),\\n            \\"high_speed_distance\\": g[\\"high_speed_dis\\"].sum(),\\n            \\"n_frames\\": g.size(),\\n        }\\n    )\\n\\n    # ---- post-snap burst -------------------------------------------------\\n    merged = chunk.merge(snap_times, on=[\\"game_id\\", \\"play_id\\"], how=\\"left\\")\\n    merged[\\"rel\\"] = (\\n        merged[\\"time\\"] - merged[\\"snap_time\\"]\\n    ).dt.total_seconds()\\n    post = merged[(merged[\\"rel\\"] >= 0) & (merged[\\"rel\\"] <= SNAP_BURST_WINDOW_S)]\\n    if not post.empty:\\n        pg = post.groupby(keys, sort=False, observed=True)\\n        burst = pd.DataFrame(\\n            {\\n                \\"snap_burst_peak_accel\\": pg[\\"a\\"].max(),\\n                \\"snap_burst_peak_speed\\": pg[\\"s\\"].max(),\\n                \\"snap_burst_distance\\": pg[\\"dis\\"].sum(),\\n                \\"snap_burst_n\\": pg.size(),\\n            }\\n        )\\n        # speed ~0.5 s after the snap: frame closest to rel == 0.5\\n        near = post.assign(_d=(post[\\"rel\\"] - 0.5).abs())\\n        idx = near.groupby(keys, sort=False, observed=True)[\\"_d\\"].idxmin()\\n        s05 = (\\n            near.loc[idx, keys + [\\"s\\"]]\\n            .rename(columns={\\"s\\": \\"snap_speed_0p5\\"})\\n            .set_index(keys)\\n        )\\n        # first tracked frame relative to the snap\\n        first = post.groupby(keys, sort=False, observed=True)[\\"rel\\"].min().rename(\\n            \\"snap_first_frame_s\\"\\n        )\\n        feats = feats.join([burst, s05, first], how=\\"left\\")\\n\\n    return feats.reset_index()\\n\\n\\ndef build_game_tracking_features(\\n    *,\\n    seasons: Sequence[int] = (2023, 2024, 2025),\\n    chunksize: int = 1_000_000,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Per-player raw in-game motion features, streamed season by season.\\n\\n    Returns one row per ``nfl_id`` with peak/typical values across all of that\\n    player\'s tracked plays, plus a play-level table cached separately.\\n    \\"\\"\\"\\n    play_frames: List[pd.DataFrame] = []\\n    for season in seasons:\\n        log.info(\\"game tracking: season %s\\", season)\\n        total = 0\\n        for chunk in io.iter_game_tracking(\\n            season, chunksize=chunksize, usecols=_GAME_TRACKING_USECOLS\\n        ):\\n            chunk[\\"time\\"] = pd.to_datetime(chunk[\\"time\\"], errors=\\"coerce\\")\\n            chunk = chunk.dropna(subset=[\\"time\\", \\"nfl_id\\"])\\n            snaps = _snap_times_for(chunk)\\n            feats = _play_features_for_chunk(chunk, snaps)\\n            feats[\\"season\\"] = season\\n            play_frames.append(feats)\\n            total += len(chunk)\\n        log.info(\\"  %s frames -> %s player-plays\\", total, sum(len(f) for f in play_frames))\\n\\n    if not play_frames:\\n        return pd.DataFrame(columns=[\\"nfl_id\\"])\\n\\n    play = pd.concat(play_frames, ignore_index=True)\\n    io.write_parquet_cache(play, paths.CACHE_INGAME_PLAY_FEATURES)\\n\\n    g = play.groupby(\\"nfl_id\\", observed=True)\\n    out = pd.DataFrame(\\n        {\\n            \\"gt_plays\\": g.size(),\\n            \\"gt_games\\": g[\\"game_id\\"].nunique(),\\n            \\"gt_peak_speed\\": g[\\"peak_speed\\"].max(),\\n            \\"gt_p90_peak_speed\\": g[\\"peak_speed\\"].quantile(0.90),\\n            \\"gt_mean_speed\\": g[\\"mean_speed\\"].mean(),\\n            \\"gt_peak_accel\\": g[\\"peak_accel\\"].max(),\\n            \\"gt_peak_decel\\": g[\\"peak_decel\\"].min(),\\n            \\"gt_distance_per_play\\": g[\\"distance\\"].mean(),\\n            \\"gt_distance_total\\": g[\\"distance\\"].sum(),\\n            \\"gt_high_speed_distance\\": g[\\"high_speed_distance\\"].sum(),\\n            \\"gt_n_frames\\": g[\\"n_frames\\"].sum(),\\n            \\"gt_snap_burst_peak_accel\\": g[\\"snap_burst_peak_accel\\"].quantile(0.90),\\n            \\"gt_snap_burst_peak_speed\\": g[\\"snap_burst_peak_speed\\"].quantile(0.90),\\n            \\"gt_snap_speed_0p5\\": g[\\"snap_speed_0p5\\"].median(),\\n            \\"gt_snap_first_frame_s\\": g[\\"snap_first_frame_s\\"].median(),\\n        }\\n    )\\n    out[\\"gt_high_speed_share\\"] = out[\\"gt_high_speed_distance\\"] / out[\\n        \\"gt_distance_total\\"\\n    ].replace(0, np.nan)\\n    out[\\"gt_high_speed_distance_per_play\\"] = out[\\"gt_high_speed_distance\\"] / out[\\n        \\"gt_plays\\"\\n    ].replace(0, np.nan)\\n    return out.reset_index()\\n\\n\\ndef build_ingame_player_features(*, force: bool = False) -> pd.DataFrame:\\n    \\"\\"\\"Combine vendor outcomes (player_play) with our own tracking features.\\"\\"\\"\\n\\n    def _build() -> pd.DataFrame:\\n        pp = build_player_play_outcomes()\\n        gt = build_game_tracking_features()\\n        return pp.merge(gt, on=\\"nfl_id\\", how=\\"outer\\")\\n\\n    return io.cached(paths.CACHE_INGAME_PLAYER_FEATURES, _build, force=force)\\n", "feasibility": "\\"\\"\\"Phase D -- feasibility scan: does a sensor trait add anything *beyond* the stopwatch?\\n\\nThe competition asks for linkages that are \\"non-obvious\\". Operationally that\\nmeans: a sensor-derived movement trait must explain in-game performance *after*\\nthe traditional Combine numbers and draft capital are already accounted for.\\nA trait that merely re-expresses the 40-yard dash is not a finding.\\n\\nThe scan is deliberately two-staged so that it is honest *and* fast:\\n\\nStage 1 (screen, full sample)\\n    Partial Spearman correlation between every (sensor trait, in-game outcome)\\n    pair, residualising both variables on the baseline covariate block\\n    (traditional Combine tests + draft capital + age + body size).\\n    p-values are corrected with Benjamini-Hochberg FDR.\\n\\nStage 2 (confirmation, cross-validated)\\n    For the strongest candidates only, fit ridge regressions with repeated\\n    K-fold cross-validation and report the *incremental* R^2 of adding the\\n    trait to the baseline block. This is the number quoted in the writeup.\\n\\nConfounds handled explicitly\\n----------------------------\\n* draft capital   -> in the baseline block for every pair\\n* position        -> by stratifying every analysis by position group\\n* exposure        -> rate / per-play outcomes preferred over counting stats\\n* multiple tests  -> BH-FDR in stage 1, and a minimum-N rule\\n* tracking noise  -> features come from smoothed signals (see kinematics.py)\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom dataclasses import dataclass\\nfrom typing import Dict, Iterable, List, Optional, Sequence, Tuple\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom . import paths, reliability as reliability_mod, schema\\n\\nlog = logging.getLogger(__name__)\\n\\ntry:\\n    from sklearn.linear_model import RidgeCV\\n    from sklearn.model_selection import KFold, cross_val_predict\\n    from sklearn.pipeline import make_pipeline\\n    from sklearn.preprocessing import StandardScaler\\n    from sklearn.metrics import r2_score\\n\\n    _HAVE_SKLEARN = True\\nexcept Exception:  # pragma: no cover\\n    _HAVE_SKLEARN = False\\n\\n# ---------------------------------------------------------------------------\\n# Configuration\\n# ---------------------------------------------------------------------------\\nMIN_N = 25                 # minimum complete cases for a pair to be reported\\nTOP_K_PER_OUTCOME = 12     # stage-2 budget per (position group, outcome)\\nFDR_ALPHA = 0.10\\nWINSOR_LIMITS = (0.01, 0.99)\\nCV_SPLITS = 5\\nCV_REPEATS = 4\\nINCREMENTAL_R2_FLOOR = 0.01  # below this, a sensor trait is \\"just the stopwatch\\"\\n# A trait measured on half an athlete\'s reps must predict the other half at\\n# least this well. Screening on reliability uses no outcome data at all, so it\\n# is a free reduction of the search space (see bdb/reliability.py).\\nRELIABILITY_FLOOR = 0.25\\n\\n# ---------------------------------------------------------------------------\\n# The baseline block = \\"the conventional scouting model without tracking data\\".\\n# It must be kept COMPACT: with n in the tens-to-low-hundreds per position\\n# group, an 11-parameter baseline eats the entire signal budget and makes every\\n# honest sensor trait look worthless (negative incremental R^2 by construction).\\n# ---------------------------------------------------------------------------\\nBASELINE_CORE: List[str] = [\\n    \\"log_draft_capital\\",   # everything teams already knew, in one number\\n    \\"age_at_draft\\",\\n    \\"combine_height\\",\\n    \\"combine_weight\\",\\n]\\n\\n# Position-relevant stopwatch tests. A corner\'s vertical jump is a different\\n# measurement than a tackle\'s, so the baseline is not identical across groups.\\nPOSITION_BASELINE_TESTS: Dict[str, List[str]] = {\\n    \\"WR\\": [\\"forty\\", \\"ten_yd_split\\", \\"three_cone\\", \\"short_shuttle\\", \\"vertical\\", \\"broad_jump\\"],\\n    \\"TE\\": [\\"forty\\", \\"ten_yd_split\\", \\"three_cone\\", \\"short_shuttle\\", \\"vertical\\", \\"broad_jump\\"],\\n    \\"DB\\": [\\"forty\\", \\"ten_yd_split\\", \\"three_cone\\", \\"short_shuttle\\", \\"vertical\\", \\"broad_jump\\"],\\n    \\"OL\\": [\\"forty\\", \\"ten_yd_split\\", \\"three_cone\\", \\"short_shuttle\\", \\"vertical\\", \\"broad_jump\\", \\"bench_reps\\"],\\n    \\"DL\\": [\\"forty\\", \\"ten_yd_split\\", \\"three_cone\\", \\"short_shuttle\\", \\"vertical\\", \\"broad_jump\\", \\"bench_reps\\"],\\n}\\n\\n# Traditional tests whose opt-out rate is high enough that the *fact of not\\n# testing* may itself be informative (injured, or protecting a bad number).\\nMISSINGNESS_FLAG_THRESHOLD = 0.10\\n\\n# Kept for backwards compatibility / the full-covariate sensitivity analysis.\\nBASELINE_COVARIATES: List[str] = BASELINE_CORE + sorted(\\n    {t for tests in POSITION_BASELINE_TESTS.values() for t in tests}\\n)\\n\\n# outcome short-name -> source column in the analysis table\\nOUTCOME_COLUMNS: Dict[str, str] = {\\n    \\"sep_mean\\": \\"sep_mean\\",\\n    \\"sep_p75\\": \\"sep_p75\\",\\n    \\"sep_max\\": \\"sep_max\\",\\n    \\"cushion_mean\\": \\"cushion_mean\\",\\n    \\"target_rate\\": \\"target_rate\\",\\n    \\"yac_mean\\": \\"yac_mean\\",\\n    \\"eyac_mean\\": \\"eyac_mean\\",\\n    \\"rec_yards_per_target\\": \\"rec_yards_per_target\\",\\n    \\"yac_per_reception\\": \\"yac_per_reception\\",\\n    \\"get_off_mean\\": \\"get_off_mean\\",\\n    \\"get_off_p25\\": \\"get_off_p25\\",\\n    \\"pressure_rate\\": \\"pressure_allowed_rate\\",\\n    \\"pressure_allowed_rate\\": \\"pressure_allowed_rate\\",\\n    \\"sack_allowed_rate\\": \\"sack_allowed_rate\\",\\n    \\"sack_rate\\": \\"sack_rate\\",\\n    \\"tfl_rate\\": \\"tfl_rate\\",\\n    \\"tackle_rate\\": \\"tackle_rate\\",\\n    \\"assist_rate\\": \\"assist_rate\\",\\n    \\"quick_pressure_rate\\": \\"quick_pressure_rate\\",\\n    \\"time_to_pressure_mean\\": \\"time_to_pressure_mean\\",\\n    \\"ttp_allowed_mean\\": \\"ttp_allowed_mean\\",\\n    \\"peak_pressure_prob_allowed\\": \\"peak_pressure_prob_allowed_mean\\",\\n    \\"dropback_duration_mean\\": \\"dropback_duration_mean\\",\\n    \\"epa_allowed_mean\\": \\"epa_mean\\",\\n    \\"epa_mean\\": \\"epa_mean\\",\\n    \\"snaps_per_season\\": \\"snaps_per_season\\",\\n    \\"games_started_per_season\\": \\"career_games_started_per_season\\",\\n    \\"honors_score\\": \\"honors_score\\",\\n}\\n\\n# Hand-curated, football-legible hypotheses: only pair a drill family with the\\n# outcomes a scout would actually care about. Keeps the scan from becoming a\\n# 10,000-cell fishing expedition.\\n_POOLED_PRIORS: Dict[str, List[str]] = {\\n    \\"WR\\": [\\"sep_mean\\", \\"sep_p75\\", \\"sep_max\\", \\"cushion_mean\\", \\"yac_per_reception\\",\\n           \\"target_rate\\"],\\n    \\"TE\\": [\\"sep_mean\\", \\"sep_p75\\", \\"cushion_mean\\", \\"yac_per_reception\\", \\"target_rate\\"],\\n    \\"DB\\": [\\"cushion_mean\\", \\"tackle_rate\\", \\"assist_rate\\", \\"epa_mean\\"],\\n    \\"OL\\": [\\"pressure_allowed_rate\\", \\"sack_allowed_rate\\", \\"ttp_allowed_mean\\",\\n           \\"peak_pressure_prob_allowed\\"],\\n    \\"DL\\": [\\"get_off_mean\\", \\"get_off_p25\\", \\"pressure_rate\\", \\"sack_rate\\",\\n           \\"quick_pressure_rate\\", \\"time_to_pressure_mean\\"],\\n}\\n\\nPOSITION_TRAIT_OUTCOME_PRIORS: Dict[str, Dict[str, List[str]]] = {\\n    \\"WR\\": {\\n        \\"pooled\\": _POOLED_PRIORS[\\"WR\\"],\\n        \\"SKILL_\\": [\\"sep_mean\\", \\"sep_p75\\", \\"cushion_mean\\", \\"yac_per_reception\\", \\"target_rate\\"],\\n        \\"three_cone\\": [\\"sep_mean\\", \\"sep_p75\\", \\"yac_per_reception\\"],\\n        \\"short_shuttle\\": [\\"sep_mean\\", \\"sep_p75\\", \\"yac_per_reception\\"],\\n        \\"forty\\": [\\"sep_mean\\", \\"target_rate\\"],\\n        \\"all_drills\\": [\\"sep_mean\\", \\"sep_p75\\", \\"target_rate\\"],\\n    },\\n    \\"TE\\": {\\n        \\"pooled\\": _POOLED_PRIORS[\\"TE\\"],\\n        \\"SKILL_\\": [\\"sep_mean\\", \\"cushion_mean\\", \\"yac_per_reception\\", \\"target_rate\\"],\\n        \\"three_cone\\": [\\"sep_mean\\", \\"yac_per_reception\\"],\\n        \\"short_shuttle\\": [\\"sep_mean\\"],\\n        \\"all_drills\\": [\\"sep_mean\\", \\"target_rate\\"],\\n    },\\n    \\"DB\\": {\\n        \\"pooled\\": _POOLED_PRIORS[\\"DB\\"],\\n        \\"SKILL_\\": [\\"cushion_mean\\", \\"tackle_rate\\", \\"assist_rate\\", \\"epa_mean\\"],\\n        \\"three_cone\\": [\\"cushion_mean\\", \\"tackle_rate\\"],\\n        \\"short_shuttle\\": [\\"cushion_mean\\", \\"tackle_rate\\"],\\n        \\"forty\\": [\\"cushion_mean\\"],\\n        \\"all_drills\\": [\\"cushion_mean\\", \\"tackle_rate\\", \\"epa_mean\\"],\\n    },\\n    \\"OL\\": {\\n        \\"pooled\\": _POOLED_PRIORS[\\"OL\\"],\\n        \\"SKILL_\\": [\\"pressure_allowed_rate\\", \\"sack_allowed_rate\\", \\"ttp_allowed_mean\\",\\n                   \\"peak_pressure_prob_allowed\\"],\\n        \\"three_cone\\": [\\"pressure_allowed_rate\\", \\"ttp_allowed_mean\\"],\\n        \\"short_shuttle\\": [\\"pressure_allowed_rate\\"],\\n        \\"all_drills\\": [\\"pressure_allowed_rate\\", \\"ttp_allowed_mean\\"],\\n    },\\n    \\"DL\\": {\\n        \\"pooled\\": _POOLED_PRIORS[\\"DL\\"],\\n        \\"SKILL_\\": [\\"get_off_mean\\", \\"get_off_p25\\", \\"pressure_rate\\", \\"sack_rate\\",\\n                   \\"tfl_rate\\", \\"quick_pressure_rate\\", \\"time_to_pressure_mean\\"],\\n        \\"forty\\": [\\"get_off_mean\\", \\"pressure_rate\\"],\\n        \\"three_cone\\": [\\"get_off_mean\\", \\"pressure_rate\\"],\\n        \\"short_shuttle\\": [\\"get_off_mean\\", \\"pressure_rate\\"],\\n        \\"all_drills\\": [\\"get_off_mean\\", \\"pressure_rate\\", \\"tfl_rate\\"],\\n    },\\n}\\n\\n\\n# ---------------------------------------------------------------------------\\n# Statistics helpers\\n# ---------------------------------------------------------------------------\\ndef bh_fdr(pvalues: np.ndarray) -> np.ndarray:\\n    \\"\\"\\"Benjamini-Hochberg adjusted p-values (q-values).\\"\\"\\"\\n    p = np.asarray(pvalues, dtype=float)\\n    ok = np.isfinite(p)\\n    q = np.full_like(p, np.nan)\\n    if not ok.any():\\n        return q\\n    pv = p[ok]\\n    n = pv.size\\n    order = np.argsort(pv)\\n    ranked = pv[order]\\n    adj = ranked * n / (np.arange(n) + 1)\\n    adj = np.minimum.accumulate(adj[::-1])[::-1]\\n    adj = np.clip(adj, 0, 1)\\n    out = np.empty(n)\\n    out[order] = adj\\n    q[ok] = out\\n    return q\\n\\n\\ndef _winsorize(s: pd.Series, limits: Tuple[float, float] = WINSOR_LIMITS) -> pd.Series:\\n    lo, hi = s.quantile(limits[0]), s.quantile(limits[1])\\n    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:\\n        return s\\n    return s.clip(lo, hi)\\n\\n\\ndef _rank(s: pd.Series) -> pd.Series:\\n    return s.rank(pct=True, na_option=\\"keep\\")\\n\\n\\ndef _residualize(y: np.ndarray, Z: np.ndarray, ridge: float = 1e-6) -> np.ndarray:\\n    \\"\\"\\"OLS residuals of y on [1, Z] with a small ridge for numerical safety.\\"\\"\\"\\n    Z = np.column_stack([np.ones(len(Z)), Z])\\n    gram = Z.T @ Z + ridge * np.eye(Z.shape[1])\\n    try:\\n        beta = np.linalg.solve(gram, Z.T @ y)\\n    except np.linalg.LinAlgError:  # pragma: no cover\\n        beta = np.linalg.lstsq(Z, y, rcond=None)[0]\\n    return y - Z @ beta\\n\\n\\ndef partial_spearman(\\n    x: pd.Series, y: pd.Series, Z: pd.DataFrame\\n) -> Tuple[float, float, int]:\\n    \\"\\"\\"Partial Spearman(x, y | Z) with a two-sided t-test p-value.\\"\\"\\"\\n    df = pd.concat([x.rename(\\"x\\"), y.rename(\\"y\\"), Z], axis=1).dropna()\\n    n = len(df)\\n    if n < MIN_N:\\n        return np.nan, np.nan, n\\n    k = Z.shape[1]\\n    if n - k - 2 < 3:\\n        return np.nan, np.nan, n\\n    rx = _rank(df[\\"x\\"]).to_numpy()\\n    ry = _rank(df[\\"y\\"]).to_numpy()\\n    Zv = df[Z.columns].to_numpy(dtype=float)\\n    ex = _residualize(rx, Zv)\\n    ey = _residualize(ry, Zv)\\n    if np.std(ex) == 0 or np.std(ey) == 0:\\n        return np.nan, np.nan, n\\n    r = float(np.corrcoef(ex, ey)[0, 1])\\n    r = float(np.clip(r, -0.999999, 0.999999))\\n    dof = n - k - 2\\n    t = r * np.sqrt(dof / (1 - r ** 2))\\n    # two-sided p from the t distribution without scipy\\n    from math import lgamma\\n\\n    def _betainc(a: float, b: float, x: float, n_iter: int = 200) -> float:\\n        \\"\\"\\"Regularised incomplete beta via continued fraction (Lentz).\\"\\"\\"\\n        if x <= 0:\\n            return 0.0\\n        if x >= 1:\\n            return 1.0\\n        lbeta = lgamma(a) + lgamma(b) - lgamma(a + b)\\n        front = np.exp(np.log(x) * a + np.log(1 - x) * b - lbeta) / a\\n        f, c, d = 1.0, 1.0, 0.0\\n        for i in range(n_iter):\\n            m = i // 2\\n            if i == 0:\\n                num = 1.0\\n            elif i % 2 == 0:\\n                num = (m * (b - m) * x) / ((a + 2 * m - 1) * (a + 2 * m))\\n            else:\\n                num = -((a + m) * (a + b + m) * x) / ((a + 2 * m) * (a + 2 * m + 1))\\n            d = 1.0 + num * d\\n            d = 1e-30 if abs(d) < 1e-30 else d\\n            d = 1.0 / d\\n            c = 1.0 + num / c\\n            c = 1e-30 if abs(c) < 1e-30 else c\\n            f *= c * d\\n            if abs(1 - c * d) < 1e-10:\\n                break\\n        return front * (f - 1.0)\\n\\n    x_beta = dof / (dof + t ** 2)\\n    p = _betainc(dof / 2.0, 0.5, x_beta)\\n    return r, float(p), n\\n\\n\\ndef cv_incremental_r2(\\n    X_base: pd.DataFrame,\\n    x_extra: pd.Series,\\n    y: pd.Series,\\n    *,\\n    splits: int = CV_SPLITS,\\n    repeats: int = CV_REPEATS,\\n    seed: int = 0,\\n) -> Dict[str, float]:\\n    \\"\\"\\"Cross-validated R^2 of baseline vs baseline+trait, and the difference.\\"\\"\\"\\n    if not _HAVE_SKLEARN:\\n        return {\\"r2_base\\": np.nan, \\"r2_full\\": np.nan, \\"delta_r2\\": np.nan, \\"n\\": 0}\\n\\n    df = pd.concat([X_base, x_extra.rename(\\"__trait\\"), y.rename(\\"__y\\")], axis=1).dropna()\\n    n = len(df)\\n    if n < MIN_N:\\n        return {\\"r2_base\\": np.nan, \\"r2_full\\": np.nan, \\"delta_r2\\": np.nan, \\"n\\": n}\\n\\n    Xb = df[X_base.columns].to_numpy(dtype=float)\\n    xt = df[[\\"__trait\\"]].to_numpy(dtype=float)\\n    yy = df[\\"__y\\"].to_numpy(dtype=float)\\n\\n    alphas = np.logspace(-2, 3, 24)\\n    n = len(yy)\\n\\n    def _cv_r2(X: np.ndarray) -> float:\\n        \\"\\"\\"R^2 on POOLED out-of-fold predictions, averaged over repeats.\\n\\n        Taking the mean of per-fold R^2 is not safe here. With n in the tens,\\n        an individual test fold can have near-zero outcome variance, which\\n        sends that fold\'s R^2 to a large negative number and makes the\\n        baseline-vs-full difference explode (observed: an impossible\\n        incremental R^2 of +3.9 before this fix). Pooling the out-of-fold\\n        predictions first, then scoring once, is stable and bounded by 1.\\n        \\"\\"\\"\\n        model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas))\\n        preds = np.zeros((repeats, n), dtype=float)\\n        for r in range(repeats):\\n            cv = KFold(n_splits=splits, shuffle=True, random_state=seed + r)\\n            try:\\n                preds[r] = cross_val_predict(model, X, yy, cv=cv)\\n            except Exception:  # pragma: no cover\\n                return float(\\"nan\\")\\n        pooled = preds.mean(axis=0)\\n        return float(r2_score(yy, pooled))\\n\\n    r2_base = _cv_r2(Xb)\\n    r2_full = _cv_r2(np.column_stack([Xb, xt]))\\n    r2_trait_only = _cv_r2(xt)\\n\\n    return {\\n        \\"r2_base\\": r2_base,\\n        \\"r2_full\\": r2_full,\\n        \\"delta_r2\\": r2_full - r2_base,\\n        \\"r2_trait_only\\": r2_trait_only,\\n        \\"n\\": n,\\n    }\\n\\n\\n# ---------------------------------------------------------------------------\\n# Trait discovery\\n# ---------------------------------------------------------------------------\\n# ``<drill>__<feature>__<stat>`` -> the drill is the part before the first \\"__\\"\\n_TRAIT_RE = None\\n\\n\\n_STAT_SUFFIXES = (\\"mean\\", \\"max\\", \\"min\\", \\"p75\\", \\"p25\\")\\n\\n\\ndef parse_trait(trait: str) -> Tuple[str, str, str]:\\n    \\"\\"\\"Split ``<drill_slug>__<feature>_<stat>`` into its three parts.\\n\\n    Note the single underscore between feature and stat and the double\\n    underscore between drill and feature -- getting this wrong silently\\n    disables the change-of-direction guard, which is exactly the bug this\\n    docstring exists to prevent.\\n    \\"\\"\\"\\n    if trait.startswith(\\"all_drills_\\"):\\n        body, drill = trait[len(\\"all_drills_\\"):], \\"all_drills\\"\\n    elif \\"__\\" in trait:\\n        drill, body = trait.split(\\"__\\", 1)\\n    else:\\n        return \\"all_drills\\", trait, \\"\\"\\n    for stat in _STAT_SUFFIXES:\\n        if body.endswith(\\"_\\" + stat):\\n            return drill, body[: -(len(stat) + 1)], stat\\n    return drill, body, \\"\\"\\n\\n\\ndef is_valid_trait(trait: str) -> bool:\\n    \\"\\"\\"Reject traits that are undefined for the drill they were computed on.\\n\\n    A change-of-direction trait extracted from a straight-line sprint is a\\n    measurement of tracking noise, not of the athlete. Letting those columns\\n    into the screen produced the single largest spurious effect on the real\\n    data, so the guard is enforced here rather than left to interpretation.\\n    \\"\\"\\"\\n    drill, feature, _ = parse_trait(trait)\\n    if feature in schema.NON_TRAIT_FEATURES:\\n        return False\\n    if drill == \\"forty_yard_dash\\" and feature in schema.COD_ONLY_FEATURES:\\n        return False\\n    return True\\n\\n\\ndef list_sensor_traits(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n) -> List[str]:\\n    \\"\\"\\"All sensor-derived columns (``<drill>__<feature>__<stat>``) in the table.\\"\\"\\"\\n    prefixes = tuple(\\n        d.lower().replace(\\"skill_drills_\\", \\"skill_\\") + \\"__\\"\\n        for d in schema.POSITION_TO_DRILLS.get(position_group, [])\\n    )\\n    traits = [\\n        c\\n        for c in analysis.columns\\n        if (\\n            (\\n                \\"__\\" in c\\n                and (\\n                    c.lower().startswith(prefixes)\\n                    or c.startswith(\\"all_drills_\\")\\n                )\\n                and is_valid_trait(c)\\n            )\\n            # pooled, player-level change-of-direction traits (100% coverage)\\n            or c.startswith(\\"pooled_\\")\\n        )\\n        and c != \\"pooled_retention_asym_low_n\\"\\n    ]\\n    return sorted(set(traits))\\n\\n\\ndef _trait_family(trait: str, position_group: str) -> Optional[str]:\\n    \\"\\"\\"Map a trait column back to a canonical drill-family key.\\n\\n    Canonical keys: ``forty``, ``three_cone``, ``short_shuttle``, ``SKILL_``,\\n    ``all_drills``. They match the keys of ``POSITION_TRAIT_OUTCOME_PRIORS``.\\n    \\"\\"\\"\\n    low = trait.lower()\\n    if low.startswith(\\"pooled_\\"):\\n        return \\"pooled\\"\\n    if low.startswith(\\"all_drills_\\"):\\n        return \\"all_drills\\"\\n    for drill in schema.POSITION_TO_DRILLS.get(position_group, []):\\n        slug = drill.lower().replace(\\"skill_drills_\\", \\"skill_\\") + \\"__\\"\\n        if low.startswith(slug):\\n            if drill == \\"SKILL_DRILLS_\\" + position_group:\\n                return \\"SKILL_\\"\\n            if drill == schema.FORTY_YARD_DASH:\\n                return \\"forty\\"\\n            if drill == schema.THREE_CONE_DRILL:\\n                return \\"three_cone\\"\\n            if drill == schema.SHORT_SHUTTLE:\\n                return \\"short_shuttle\\"\\n            return drill.lower()\\n    return None\\n\\n\\ndef _prior_allows(position_group: str, family: Optional[str], outcome: str) -> Tuple[bool, bool]:\\n    \\"\\"\\"Return (allowed, is_confirmatory) for a (family, outcome) pair.\\"\\"\\"\\n    priors = POSITION_TRAIT_OUTCOME_PRIORS.get(position_group, {})\\n    if not priors or family is None or family not in priors:\\n        return True, False\\n    return (outcome in set(priors[family])), True\\n\\n\\ndef baseline_columns_for(position_group: str, available: Sequence[str]) -> List[str]:\\n    \\"\\"\\"The compact, position-appropriate conventional-scouting baseline.\\"\\"\\"\\n    desired = BASELINE_CORE + POSITION_BASELINE_TESTS.get(position_group, [])\\n    seen, cols = set(), []\\n    for c in desired:\\n        if c in available and c not in seen:\\n            seen.add(c)\\n            cols.append(c)\\n    return cols\\n\\n\\ndef impute_baseline(\\n    Z: pd.DataFrame,\\n    *,\\n    flag_threshold: float = MISSINGNESS_FLAG_THRESHOLD,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Median-impute the baseline and flag the heavily-missing tests.\\n\\n    Complete-case deletion is not an option here: with Combine opt-out rates of\\n    20-35% on the agility tests, requiring every traditional number would throw\\n    away more than half of an already small cohort. Imputing the median and\\n    carrying an explicit \\"did not test\\" indicator keeps the sample while making\\n    the imputation visible to the model (an athlete who skipped the 3-cone is\\n    not interchangeable with one who ran it).\\n    \\"\\"\\"\\n    Z = Z.apply(pd.to_numeric, errors=\\"coerce\\")\\n    out = Z.copy()\\n    medians = Z.median(numeric_only=True)\\n    for col in Z.columns:\\n        miss = Z[col].isna()\\n        rate = float(miss.mean())\\n        if rate > flag_threshold:\\n            out[f\\"{col}_not_tested\\"] = miss.astype(float)\\n        if Z[col].notna().any():\\n            out[col] = Z[col].fillna(medians[col])\\n        else:\\n            out[col] = 0.0\\n    return out\\n\\n\\ndef build_design(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    outcome: str,\\n) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, List[str]]:\\n    \\"\\"\\"Return (baseline covariates, outcome, full frame, trait columns).\\"\\"\\"\\n    sub = analysis[analysis[\\"position_group\\"] == position_group].copy()\\n    source = OUTCOME_COLUMNS.get(outcome, outcome)\\n    if source not in sub.columns:\\n        raise KeyError(f\\"outcome column {source!r} not present\\")\\n    y = pd.to_numeric(sub[source], errors=\\"coerce\\")\\n    base_cols = baseline_columns_for(position_group, sub.columns)\\n    Z = impute_baseline(sub[base_cols])\\n    return Z, y, sub, list_sensor_traits(sub, position_group)\\n\\n\\n\\ndef trait_base_feature(trait: str) -> Optional[str]:\\n    \\"\\"\\"Per-cut feature that a player-level ``trait`` column is aggregated from.\\n\\n    ``skill_dl__entry_speed_max`` -> ``entry_speed`` and\\n    ``pooled_retention_mean`` -> ``speed_retention``. Exposed so that reporting\\n    code resolves reliability exactly the way the screen does, instead of\\n    failing to find an aggregate column in the per-cut reliability table.\\n    \\"\\"\\"\\n    if trait.startswith(\\"pooled_\\"):\\n        base = trait[len(\\"pooled_\\"):]\\n        for suffix in (\\"_mean\\", \\"_best\\", \\"_worst\\", \\"_angle_adj\\", \\"_rate\\"):\\n            if base.endswith(suffix):\\n                base = base[: -len(suffix)]\\n        if base == \\"retention\\":\\n            base = \\"speed_retention\\"\\n        return base\\n    try:\\n        _, feature, _ = parse_trait(trait)\\n    except Exception:  # noqa: BLE001\\n        return None\\n    return feature\\n\\n\\ndef trait_reliability(trait: str, lookup: Dict[str, float]) -> Optional[float]:\\n    \\"\\"\\"Reliability of the per-cut feature underlying a trait column.\\"\\"\\"\\n    if not lookup:\\n        return None\\n    base = trait_base_feature(trait)\\n    if base is None:\\n        return None\\n    return lookup.get(base)\\n\\n\\ndef scan_position_group(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    *,\\n    outcomes: Optional[Sequence[str]] = None,\\n    top_k: int = TOP_K_PER_OUTCOME,\\n    run_stage2: bool = True,\\n    reliability_floor: Optional[float] = RELIABILITY_FLOOR,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Stage 1 + stage 2 scan for one position group.\\n\\n    ``reliability_floor`` removes traits that are not repeatable across an\\n    athlete\'s own reps, using no outcome information whatsoever.\\n    \\"\\"\\"\\n    reliability_lookup = reliability_mod.reliability_lookup()\\n    if reliability_lookup:\\n        log.info(\\n            \\"reliability screening active: %s features known, floor=%.2f\\",\\n            len(reliability_lookup), reliability_floor if reliability_floor else 0.0,\\n        )\\n    else:\\n        log.warning(\\"no reliability table found; screening without it\\")\\n    outcomes = list(outcomes or schema.POSITION_TO_OUTCOMES.get(position_group, []))\\n    rows: List[Dict[str, object]] = []\\n\\n    for outcome in outcomes:\\n        try:\\n            Z, y, sub, traits = build_design(analysis, position_group, outcome)\\n        except KeyError as exc:\\n            log.warning(\\"skipping outcome %s: %s\\", outcome, exc)\\n            continue\\n        if y.notna().sum() < MIN_N:\\n            continue\\n\\n        # winsorise covariates once\\n        Z = Z.apply(_winsorize)\\n\\n        for trait in traits:\\n            family = _trait_family(trait, position_group)\\n            allowed, confirmatory = _prior_allows(position_group, family, outcome)\\n            if not allowed:\\n                continue\\n            rel = trait_reliability(trait, reliability_lookup)\\n            if reliability_floor is not None and rel is not None and rel < reliability_floor:\\n                continue\\n            x = pd.to_numeric(sub[trait], errors=\\"coerce\\")\\n            if x.notna().sum() < MIN_N or x.nunique(dropna=True) < 5:\\n                continue\\n            rho, p, n = partial_spearman(_winsorize(x), y, Z)\\n            if not np.isfinite(rho):\\n                continue\\n            rows.append(\\n                {\\n                    \\"position_group\\": position_group,\\n                    \\"outcome\\": outcome,\\n                    \\"trait\\": trait,\\n                    \\"drill_family\\": family,\\n                    \\"confirmatory\\": confirmatory,\\n                    \\"trait_reliability\\": rel,\\n                    \\"disattenuated_ceiling\\": (\\n                        reliability_mod.unreliability_ceiling(rel) if rel is not None else None\\n                    ),\\n                    \\"n\\": n,\\n                    \\"partial_spearman\\": rho,\\n                    \\"abs_rho\\": abs(rho),\\n                    \\"p_value\\": p,\\n                }\\n            )\\n\\n    res = pd.DataFrame(rows)\\n    if res.empty:\\n        return res\\n\\n    res[\\"q_value\\"] = np.nan\\n    for outcome, idx in res.groupby(\\"outcome\\").groups.items():\\n        res.loc[idx, \\"q_value\\"] = bh_fdr(res.loc[idx, \\"p_value\\"].to_numpy())\\n    res[\\"passes_fdr\\"] = res[\\"q_value\\"] < FDR_ALPHA\\n\\n    if not run_stage2:\\n        return res.sort_values(\\"abs_rho\\", ascending=False).reset_index(drop=True)\\n\\n    # ---------------- stage 2: cross-validated incremental R^2 ------------\\n    confirm: List[Dict[str, object]] = []\\n    for outcome, grp in res.groupby(\\"outcome\\"):\\n        Z, y, sub, _ = build_design(analysis, position_group, outcome)\\n        Z = Z.apply(_winsorize)\\n        candidates = (\\n            grp.sort_values(\\"abs_rho\\", ascending=False).head(top_k)[\\"trait\\"].tolist()\\n        )\\n        for trait in candidates:\\n            x = pd.to_numeric(sub[trait], errors=\\"coerce\\")\\n            stats = cv_incremental_r2(Z, _winsorize(x), y)\\n            row = res[(res[\\"outcome\\"] == outcome) & (res[\\"trait\\"] == trait)].iloc[0]\\n            confirm.append({**row.to_dict(), **stats})\\n    out = pd.DataFrame(confirm)\\n    if out.empty:\\n        return res\\n\\n    # FDR again, now over the confirmed stage-2 set\\n    # BH across the *whole* confirmed set for this position group, not within\\n    # each outcome: controlling per outcome would let a family of weak\\n    # candidates share a lenient correction.\\n    out[\\"cv_q_value\\"] = bh_fdr(np.nan_to_num(out[\\"p_value\\"].to_numpy(), nan=1.0))\\n    # A stage-2 claim requires the stage-1 screen to have passed too. Requiring\\n    # only the (few-candidate, therefore lenient) stage-2 correction produced\\n    # \\"survivors\\" whose stage-1 q was 0.95 -- the two stages disagreeing is a\\n    # sign of an unstable pair, not of a finding.\\n    out[\\"adds_beyond_stopwatch\\"] = (\\n        (out[\\"delta_r2\\"] > INCREMENTAL_R2_FLOOR)\\n        & (out[\\"cv_q_value\\"] < FDR_ALPHA)\\n        & (out[\\"q_value\\"] < FDR_ALPHA)\\n    )\\n    return out.sort_values(\\"delta_r2\\", ascending=False).reset_index(drop=True)\\n\\n\\ndef scan_all(\\n    analysis: pd.DataFrame,\\n    *,\\n    position_groups: Optional[Iterable[str]] = None,\\n    run_stage2: bool = True,\\n    reliability_floor: Optional[float] = RELIABILITY_FLOOR,\\n) -> pd.DataFrame:\\n    groups = list(position_groups or schema.FOCUS_POSITION_GROUPS)\\n    frames = []\\n    for group in groups:\\n        log.info(\\"feasibility scan: %s\\", group)\\n        res = scan_position_group(\\n            analysis, group, run_stage2=run_stage2, reliability_floor=reliability_floor\\n        )\\n        if not res.empty:\\n            frames.append(res)\\n    if not frames:\\n        return pd.DataFrame()\\n    return pd.concat(frames, ignore_index=True)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Trait redundancy pruning + reliability\\n# ---------------------------------------------------------------------------\\ndef prune_redundant_traits(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    *,\\n    threshold: float = 0.95,\\n) -> List[str]:\\n    \\"\\"\\"Drop near-duplicate sensor traits (|r| > threshold), keeping the first.\\"\\"\\"\\n    traits = list_sensor_traits(analysis, position_group)\\n    sub = analysis[analysis[\\"position_group\\"] == position_group]\\n    if len(sub) < 3:\\n        return traits\\n    X = sub[traits].apply(pd.to_numeric, errors=\\"coerce\\")\\n    X = X.loc[:, X.notna().sum() >= MIN_N]\\n    if X.shape[1] < 2:\\n        return list(X.columns)\\n    corr = X.corr().abs()\\n    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))\\n    drop = [c for c in upper.columns if (upper[c] > threshold).any()]\\n    return [c for c in X.columns if c not in drop]\\n\\n\\ndef split_half_reliability(\\n    attempts: pd.DataFrame,\\n    *,\\n    feature: str = \\"speed_retention\\",\\n    drill_type: str = \\"THREE_CONE_DRILL\\",\\n) -> Dict[str, float]:\\n    \\"\\"\\"Odd-vs-even attempt reliability of a per-attempt feature.\\n\\n    A trait that is not even reliable within the same player\'s own Combine\\n    attempts cannot plausibly predict NFL performance; this bounds how much\\n    signal any correlation can carry (attenuation).\\n    \\"\\"\\"\\n    sub = attempts[attempts[\\"drill_type\\"] == drill_type].copy()\\n    if sub.empty or feature not in sub.columns:\\n        return {\\"reliability\\": np.nan, \\"n_players\\": 0}\\n    sub[\\"parity\\"] = sub[\\"attempt\\"].astype(\\"Int64\\") % 2\\n    odd = sub[sub[\\"parity\\"] == 1].groupby(\\"nfl_id\\")[feature].mean()\\n    even = sub[sub[\\"parity\\"] == 0].groupby(\\"nfl_id\\")[feature].mean()\\n    joined = pd.concat([odd.rename(\\"odd\\"), even.rename(\\"even\\")], axis=1).dropna()\\n    if len(joined) < 10:\\n        return {\\"reliability\\": np.nan, \\"n_players\\": len(joined)}\\n    r = float(joined.corr().iloc[0, 1])\\n    # Spearman-Brown correction to full-length reliability\\n    rho_full = 2 * r / (1 + r) if r > -1 else np.nan\\n    return {\\"reliability\\": rho_full, \\"split_half_r\\": r, \\"n_players\\": len(joined)}\\n", "stats": "\\"\\"\\"Reusable statistics for small-N, confounded sports analytics.\\n\\nEverything here is written for the sample sizes this competition actually has:\\ntens of players per position group, heavy confounding by draft capital, and\\noutcome variables that are themselves estimated (Next Gen Stats model outputs).\\nThat rules out most of the standard toolbox and forces:\\n\\n* **bootstrap** rather than asymptotic confidence intervals,\\n* **robust (HC3) standard errors** rather than classical OLS ones,\\n* **leave-one-out / influence diagnostics** as a first-class output rather than\\n  an afterthought, because with n ~ 60 a single player can create or destroy an\\n  apparent effect,\\n* **explicit reporting of `n`** on every number that leaves this module.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport logging\\nfrom dataclasses import dataclass, field\\nfrom typing import Callable, Dict, List, Optional, Sequence, Tuple\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nlog = logging.getLogger(__name__)\\n\\n\\n# ---------------------------------------------------------------------------\\n# Rank / residual helpers\\n# ---------------------------------------------------------------------------\\ndef rank_pct(s: pd.Series) -> pd.Series:\\n    return s.rank(pct=True, na_option=\\"keep\\")\\n\\n\\ndef ols_residuals(y: np.ndarray, Z: np.ndarray, ridge: float = 1e-8) -> np.ndarray:\\n    \\"\\"\\"Residuals of ``y`` on ``[1, Z]`` using a tiny ridge for stability.\\"\\"\\"\\n    if Z.ndim == 1:\\n        Z = Z[:, None]\\n    design = np.column_stack([np.ones(len(Z)), Z])\\n    gram = design.T @ design + ridge * np.eye(design.shape[1])\\n    try:\\n        beta = np.linalg.solve(gram, design.T @ y)\\n    except np.linalg.LinAlgError:  # pragma: no cover\\n        beta = np.linalg.lstsq(design, y, rcond=None)[0]\\n    return y - design @ beta\\n\\n\\n@dataclass\\nclass OLSPointEstimate:\\n    \\"\\"\\"OLS fit with heteroskedasticity-robust (HC3) inference.\\"\\"\\"\\n\\n    coefficients: pd.Series\\n    robust_se: pd.Series\\n    t_stats: pd.Series\\n    p_values: pd.Series\\n    ci_low: pd.Series\\n    ci_high: pd.Series\\n    r_squared: float\\n    adj_r_squared: float\\n    n: int\\n    k: int\\n\\n    def to_frame(self) -> pd.DataFrame:\\n        return pd.DataFrame(\\n            {\\n                \\"coef\\": self.coefficients,\\n                \\"robust_se\\": self.robust_se,\\n                \\"t\\": self.t_stats,\\n                \\"p\\": self.p_values,\\n                \\"ci_low\\": self.ci_low,\\n                \\"ci_high\\": self.ci_high,\\n            }\\n        )\\n\\n    def summary(self) -> str:\\n        lines = [\\n            f\\"OLS with HC3 robust SE   n={self.n}  k={self.k}  \\"\\n            f\\"R2={self.r_squared:.3f}  adjR2={self.adj_r_squared:.3f}\\",\\n        ]\\n        frame = self.to_frame()\\n        for name, row in frame.iterrows():\\n            lines.append(\\n                f\\"  {name:<28} {row[\'coef\']:+.4f} \\"\\n                f\\"[{row[\'ci_low\']:+.4f}, {row[\'ci_high\']:+.4f}]  p={row[\'p\']:.4f}\\"\\n            )\\n        return \\"\\\\n\\".join(lines)\\n\\n\\ndef _t_sf(t: np.ndarray | float, dof: int) -> np.ndarray:\\n    \\"\\"\\"Two-sided p-value of Student-t, via the incomplete beta function.\\"\\"\\"\\n    from math import lgamma\\n\\n    t = np.atleast_1d(np.asarray(t, dtype=float))\\n    x = dof / (dof + t ** 2)\\n\\n    def betainc(a: float, b: float, xx: np.ndarray) -> np.ndarray:\\n        out = np.empty_like(xx)\\n        for i, v in enumerate(xx):\\n            if v <= 0:\\n                out[i] = 0.0\\n                continue\\n            if v >= 1:\\n                out[i] = 1.0\\n                continue\\n            lbeta = lgamma(a) + lgamma(b) - lgamma(a + b)\\n            front = np.exp(np.log(v) * a + np.log(1 - v) * b - lbeta) / a\\n            f, c, d = 1.0, 1.0, 0.0\\n            for j in range(300):\\n                m = j // 2\\n                if j == 0:\\n                    num = 1.0\\n                elif j % 2 == 0:\\n                    num = (m * (b - m) * v) / ((a + 2 * m - 1) * (a + 2 * m))\\n                else:\\n                    num = -((a + m) * (a + b + m) * v) / ((a + 2 * m) * (a + 2 * m + 1))\\n                d = 1.0 + num * d\\n                d = 1e-30 if abs(d) < 1e-30 else d\\n                d = 1.0 / d\\n                c = 1.0 + num / c\\n                c = 1e-30 if abs(c) < 1e-30 else c\\n                f *= c * d\\n                if abs(1 - c * d) < 1e-12:\\n                    break\\n            out[i] = front * (f - 1.0)\\n        return out\\n\\n    return betainc(dof / 2.0, 0.5, x)\\n\\n\\ndef ols_robust(\\n    X: pd.DataFrame,\\n    y: pd.Series,\\n    *,\\n    add_intercept: bool = True,\\n) -> OLSPointEstimate:\\n    \\"\\"\\"OLS with HC3 heteroskedasticity-robust standard errors.\\"\\"\\"\\n    frame = pd.concat([X, y.rename(\\"__y\\")], axis=1).dropna()\\n    n = len(frame)\\n    if n <= X.shape[1] + 2:\\n        raise ValueError(f\\"not enough complete cases (n={n}) for {X.shape[1]} predictors\\")\\n    design = frame[X.columns].to_numpy(dtype=float)\\n    if add_intercept:\\n        design = np.column_stack([np.ones(n), design])\\n        names = [\\"intercept\\"] + list(X.columns)\\n    else:  # pragma: no cover\\n        names = list(X.columns)\\n    yy = frame[\\"__y\\"].to_numpy(dtype=float)\\n\\n    xtx_inv = np.linalg.pinv(design.T @ design)\\n    beta = xtx_inv @ design.T @ yy\\n    resid = yy - design @ beta\\n    k = design.shape[1]\\n    dof = max(n - k, 1)\\n\\n    # HC3: (X\'X)^-1 X\' diag(e_i^2 / (1-h_ii)^2) X (X\'X)^-1\\n    hat = np.einsum(\\"ij,jk,ik->i\\", design, xtx_inv, design)\\n    hat = np.clip(hat, 0, 1 - 1e-8)\\n    weights = (resid ** 2) / (1.0 - hat) ** 2\\n    meat = design.T @ (design * weights[:, None])\\n    cov = xtx_inv @ meat @ xtx_inv\\n\\n    se = np.sqrt(np.clip(np.diag(cov), 0, None))\\n    t = beta / np.where(se > 0, se, np.nan)\\n    p = _t_sf(t, dof)\\n    # 95% CI via the t quantile approximated by the normal for dof > 30, else\\n    # a conservative 2.1 multiplier; exact quantiles are not worth a scipy dep here\\n    mult = np.where(dof > 60, 1.96, np.where(dof > 30, 2.04, 2.15))\\n    ss_tot = float(np.sum((yy - yy.mean()) ** 2))\\n    r2 = 1.0 - float(np.sum(resid ** 2)) / ss_tot if ss_tot > 0 else np.nan\\n    adj = 1 - (1 - r2) * (n - 1) / max(n - k, 1) if np.isfinite(r2) else np.nan\\n\\n    return OLSPointEstimate(\\n        coefficients=pd.Series(beta, index=names),\\n        robust_se=pd.Series(se, index=names),\\n        t_stats=pd.Series(t, index=names),\\n        p_values=pd.Series(p, index=names),\\n        ci_low=pd.Series(beta - mult * se, index=names),\\n        ci_high=pd.Series(beta + mult * se, index=names),\\n        r_squared=float(r2),\\n        adj_r_squared=float(adj),\\n        n=n,\\n        k=k,\\n    )\\n\\n\\n# ---------------------------------------------------------------------------\\n# Partial correlation + bootstrap\\n# ---------------------------------------------------------------------------\\ndef partial_correlation(\\n    x: pd.Series,\\n    y: pd.Series,\\n    Z: pd.DataFrame,\\n    *,\\n    rank: bool = True,\\n) -> Tuple[float, int]:\\n    \\"\\"\\"Partial correlation of x and y given Z (Spearman if ``rank``).\\"\\"\\"\\n    frame = pd.concat([x.rename(\\"x\\"), y.rename(\\"y\\"), Z], axis=1).dropna()\\n    n = len(frame)\\n    if n < 5:\\n        return np.nan, n\\n    xv = rank_pct(frame[\\"x\\"]).to_numpy() if rank else frame[\\"x\\"].to_numpy()\\n    yv = rank_pct(frame[\\"y\\"]).to_numpy() if rank else frame[\\"y\\"].to_numpy()\\n    Zv = frame[Z.columns].to_numpy(dtype=float)\\n    ex = ols_residuals(xv, Zv)\\n    ey = ols_residuals(yv, Zv)\\n    if np.std(ex) == 0 or np.std(ey) == 0:\\n        return np.nan, n\\n    return float(np.corrcoef(ex, ey)[0, 1]), n\\n\\n\\ndef bootstrap_partial_correlation(\\n    x: pd.Series,\\n    y: pd.Series,\\n    Z: pd.DataFrame,\\n    *,\\n    n_boot: int = 2000,\\n    alpha: float = 0.05,\\n    seed: int = 0,\\n    rank: bool = True,\\n    cluster: Optional[pd.Series] = None,\\n) -> Dict[str, float]:\\n    \\"\\"\\"Percentile bootstrap CI for the partial correlation.\\n\\n    If ``cluster`` is supplied (e.g. draft class), whole clusters are resampled,\\n    which is the right thing to do when players within a season share shocks.\\n    \\"\\"\\"\\n    frame = pd.concat(\\n        [x.rename(\\"x\\"), y.rename(\\"y\\"), Z], axis=1\\n    ).dropna()\\n    if cluster is not None:\\n        frame = frame.join(cluster.rename(\\"__cluster\\")).dropna(subset=[\\"__cluster\\"])\\n    n = len(frame)\\n    if n < 8:\\n        return {\\"estimate\\": np.nan, \\"ci_low\\": np.nan, \\"ci_high\\": np.nan,\\n                \\"n\\": n, \\"n_boot\\": 0, \\"p_boot\\": np.nan}\\n\\n    est, _ = partial_correlation(frame[\\"x\\"], frame[\\"y\\"], frame[Z.columns], rank=rank)\\n\\n    rng = np.random.default_rng(seed)\\n    stats: List[float] = []\\n    if cluster is not None:\\n        groups = frame.groupby(\\"__cluster\\").indices\\n        keys = list(groups)\\n        for _ in range(n_boot):\\n            pick = rng.choice(keys, size=len(keys), replace=True)\\n            idx = np.concatenate([groups[k] for k in pick])\\n            sub = frame.iloc[idx]\\n            r, _ = partial_correlation(sub[\\"x\\"], sub[\\"y\\"], sub[Z.columns], rank=rank)\\n            if np.isfinite(r):\\n                stats.append(r)\\n    else:\\n        pos = np.arange(n)\\n        for _ in range(n_boot):\\n            idx = rng.choice(pos, size=n, replace=True)\\n            sub = frame.iloc[idx]\\n            r, _ = partial_correlation(sub[\\"x\\"], sub[\\"y\\"], sub[Z.columns], rank=rank)\\n            if np.isfinite(r):\\n                stats.append(r)\\n\\n    if not stats:\\n        return {\\"estimate\\": est, \\"ci_low\\": np.nan, \\"ci_high\\": np.nan,\\n                \\"n\\": n, \\"n_boot\\": 0, \\"p_boot\\": np.nan}\\n    arr = np.asarray(stats)\\n    lo, hi = np.quantile(arr, [alpha / 2, 1 - alpha / 2])\\n    # bootstrap two-sided p: fraction of resamples on the wrong side of zero\\n    p_boot = 2.0 * min((arr <= 0).mean(), (arr >= 0).mean())\\n    return {\\n        \\"estimate\\": float(est),\\n        \\"ci_low\\": float(lo),\\n        \\"ci_high\\": float(hi),\\n        \\"n\\": n,\\n        \\"n_boot\\": len(arr),\\n        \\"p_boot\\": float(min(p_boot, 1.0)),\\n    }\\n\\n\\n# ---------------------------------------------------------------------------\\n# Influence / robustness\\n# ---------------------------------------------------------------------------\\ndef leave_one_out_stability(\\n    x: pd.Series,\\n    y: pd.Series,\\n    Z: pd.DataFrame,\\n    *,\\n    rank: bool = True,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Recompute the partial correlation with each observation removed.\\"\\"\\"\\n    frame = pd.concat([x.rename(\\"x\\"), y.rename(\\"y\\"), Z], axis=1).dropna()\\n    base, _ = partial_correlation(frame[\\"x\\"], frame[\\"y\\"], Z, rank=rank)\\n    rows = []\\n    for i in range(len(frame)):\\n        sub = frame.drop(frame.index[i])\\n        r, _ = partial_correlation(sub[\\"x\\"], sub[\\"y\\"], Z, rank=rank)\\n        rows.append({\\"dropped_index\\": frame.index[i], \\"partial_r\\": r, \\"base_r\\": base,\\n                     \\"delta\\": r - base if np.isfinite(r) else np.nan})\\n    out = pd.DataFrame(rows)\\n    # attach the player identity when available\\n    if \\"nfl_id\\" in frame.columns:\\n        out[\\"nfl_id\\"] = frame[\\"nfl_id\\"].to_numpy()\\n    return out\\n\\n\\ndef influence_summary(loo: pd.DataFrame) -> Dict[str, float]:\\n    if loo.empty:\\n        return {\\"max_abs_delta\\": np.nan, \\"n_flips_sign\\": 0, \\"base_r\\": np.nan}\\n    base = float(loo[\\"base_r\\"].iloc[0])\\n    flips = int(((loo[\\"partial_r\\"] > 0) != (base > 0)).sum())\\n    return {\\n        \\"base_r\\": base,\\n        \\"max_abs_delta\\": float(loo[\\"delta\\"].abs().max()),\\n        \\"min_partial_r\\": float(loo[\\"partial_r\\"].min()),\\n        \\"max_partial_r\\": float(loo[\\"partial_r\\"].max()),\\n        \\"n_flips_sign\\": flips,\\n        \\"n\\": int(len(loo)),\\n    }\\n\\n\\n# ---------------------------------------------------------------------------\\n# Metric construction\\n# ---------------------------------------------------------------------------\\ndef zscore_within(s: pd.Series, groups: Optional[pd.Series] = None) -> pd.Series:\\n    \\"\\"\\"Standardise, optionally within position group, ignoring missing values.\\"\\"\\"\\n    if groups is None:\\n        return (s - s.mean()) / s.std(ddof=1)\\n    g = s.groupby(groups)\\n    return (s - g.transform(\\"mean\\")) / g.transform(\\"std\\")\\n\\n\\ndef composite_metric(\\n    frame: pd.DataFrame,\\n    components: Sequence[str],\\n    *,\\n    direction: Sequence[float],\\n    weights: Optional[Sequence[float]] = None,\\n    group: Optional[str] = None,\\n) -> pd.Series:\\n    \\"\\"\\"Percentile-average composite, standardised within position group.\\n\\n    ``direction`` carries ``+1`` for \\"higher is better\\" and ``-1`` for the\\n    reverse, so that a metric such as \\"Exit Velocity Ratio\\" (higher better) can\\n    be combined with \\"time to regain speed\\" (lower better).\\n    \\"\\"\\"\\n    if weights is None:\\n        weights = [1.0] * len(components)\\n    w = np.asarray(weights, dtype=float)\\n    w = w / w.sum()\\n    groups = frame[group] if group and group in frame.columns else None\\n\\n    total = None\\n    for comp, sgn, weight in zip(components, direction, w):\\n        if comp not in frame.columns:\\n            raise KeyError(f\\"composite component missing: {comp}\\")\\n        z = zscore_within(pd.to_numeric(frame[comp], errors=\\"coerce\\"), groups) * sgn\\n        total = z * weight if total is None else total + z * weight\\n    return total\\n\\n\\ndef metric_leaderboard(\\n    frame: pd.DataFrame,\\n    metric: pd.Series,\\n    *,\\n    position_group: Optional[str] = None,\\n    top: int = 15,\\n    columns: Optional[Sequence[str]] = None,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Rank players by the metric, optionally inside one position group.\\"\\"\\"\\n    sub = frame.copy()\\n    sub[\\"_metric\\"] = metric\\n    if position_group and \\"position_group\\" in sub.columns:\\n        sub = sub[sub[\\"position_group\\"] == position_group]\\n    cols = [\\"nfl_id\\"]\\n    for c in (columns or [\\"display_name\\", \\"position_group\\", \\"draft_overall_pick\\"]):\\n        if c in sub.columns:\\n            cols.append(c)\\n    cols.append(\\"_metric\\")\\n    return (\\n        sub.dropna(subset=[\\"_metric\\"])\\n        .sort_values(\\"_metric\\", ascending=False)[cols]\\n        .head(top)\\n        .reset_index(drop=True)\\n    )\\n\\n\\ndef quadrant_split(\\n    frame: pd.DataFrame,\\n    sensor_metric: pd.Series,\\n    traditional_metric: pd.Series,\\n    *,\\n    sensor_label: str = \\"sensor\\",\\n    traditional_label: str = \\"stopwatch\\",\\n) -> pd.DataFrame:\\n    \\"\\"\\"Classify players into the four scouting archetypes.\\n\\n    The interesting quadrants are the two *disagreements*: players the sensor\\n    likes but the stopwatch does not (candidate hidden talent) and vice versa\\n    (candidate over-drafted profile).\\n    \\"\\"\\"\\n    sub = frame.copy()\\n    sub[\\"_sensor\\"] = pd.to_numeric(sensor_metric, errors=\\"coerce\\")\\n    sub[\\"_trad\\"] = pd.to_numeric(traditional_metric, errors=\\"coerce\\")\\n    sub = sub.dropna(subset=[\\"_sensor\\", \\"_trad\\"])\\n    if sub.empty:\\n        return sub\\n    s_med = sub[\\"_sensor\\"].median()\\n    t_med = sub[\\"_trad\\"].median()\\n\\n    def _label(row: pd.Series) -> str:\\n        hi_s, hi_t = row[\\"_sensor\\"] >= s_med, row[\\"_trad\\"] >= t_med\\n        if hi_s and hi_t:\\n            return \\"confirmed_elite\\"\\n        if hi_s and not hi_t:\\n            return f\\"{sensor_label}_beats_{traditional_label}\\"\\n        if not hi_s and hi_t:\\n            return f\\"{traditional_label}_beats_{sensor_label}\\"\\n        return \\"confirmed_limited\\"\\n\\n    sub[\\"archetype\\"] = sub.apply(_label, axis=1)\\n    sub[\\"sensor_z\\"] = (sub[\\"_sensor\\"] - sub[\\"_sensor\\"].mean()) / sub[\\"_sensor\\"].std(ddof=1)\\n    sub[\\"trad_z\\"] = (sub[\\"_trad\\"] - sub[\\"_trad\\"].mean()) / sub[\\"_trad\\"].std(ddof=1)\\n    sub[\\"disagreement\\"] = sub[\\"sensor_z\\"] - sub[\\"trad_z\\"]\\n    return sub\\n\\n\\ndef rank_stability(\\n    frame: pd.DataFrame,\\n    metric_fn: Callable[[pd.DataFrame], pd.Series],\\n    *,\\n    n_boot: int = 500,\\n    seed: int = 0,\\n    top: int = 10,\\n) -> Dict[str, float]:\\n    \\"\\"\\"How stable is the top-k of a metric under bootstrap resampling?\\n\\n    A metric whose leaderboard reshuffles completely under resampling is not\\n    something a front office can act on, however significant its aggregate\\n    correlation looks.\\n    \\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n    pos = np.arange(len(frame))\\n    tops: List[set] = []\\n    for _ in range(n_boot):\\n        idx = rng.choice(pos, size=len(frame), replace=True)\\n        sample = frame.iloc[idx].copy()\\n        try:\\n            m = metric_fn(sample)\\n        except Exception:  # pragma: no cover\\n            continue\\n        ranked = m.dropna().sort_values(ascending=False)\\n        tops.append(set(ranked.head(top).index))\\n    if not tops:\\n        return {\\"mean_jaccard\\": np.nan, \\"n_boot\\": 0}\\n    jaccards = []\\n    for i in range(len(tops)):\\n        for j in range(i + 1, len(tops)):\\n            a, b = tops[i], tops[j]\\n            if not a and not b:\\n                continue\\n            jaccards.append(len(a & b) / max(len(a | b), 1))\\n    return {\\n        \\"mean_jaccard\\": float(np.mean(jaccards)) if jaccards else np.nan,\\n        \\"median_jaccard\\": float(np.median(jaccards)) if jaccards else np.nan,\\n        \\"n_boot\\": len(tops),\\n    }\\n", "deepdive": "\\"\\"\\"Phases E/F -- confirmatory analysis and metric construction.\\n\\nThe feasibility scan is a *screen*: it deliberately looks at many pairs and can\\nonly tell us where to look. This module is the confirmation step for the one\\ntopic that survives, and it is written to make over-claiming difficult:\\n\\n* every effect is reported with a bootstrap confidence interval and its n;\\n* every effect is re-estimated under leave-one-out to expose single-player\\n  dependence;\\n* the baseline is always the conventional scouting model, so the headline claim\\n  is an *incremental* claim;\\n* the metric that gets published is checked for rank stability, because a metric\\n  whose leaderboard reshuffles under resampling is not actionable even if its\\n  aggregate correlation is significant.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nimport logging\\nfrom dataclasses import dataclass, field\\nfrom typing import Any, Dict, List, Optional, Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom . import feasibility, io, paths, stats\\n\\nlog = logging.getLogger(__name__)\\n\\n\\n@dataclass\\nclass LinkageResult:\\n    \\"\\"\\"Everything needed to defend one Combine -> NFL linkage claim.\\"\\"\\"\\n\\n    position_group: str\\n    outcome: str\\n    trait: str\\n    partial_r: float\\n    ci_low: float\\n    ci_high: float\\n    n: int\\n    p_boot: float\\n    q_value: float\\n    delta_r2: float\\n    r2_base: float\\n    r2_full: float\\n    reliability: Optional[float] = None\\n    loo: Dict[str, float] = field(default_factory=dict)\\n    ols: Optional[str] = None\\n    extras: Dict[str, Any] = field(default_factory=dict)\\n\\n    def to_dict(self) -> Dict[str, Any]:\\n        payload = {\\n            \\"position_group\\": self.position_group,\\n            \\"outcome\\": self.outcome,\\n            \\"trait\\": self.trait,\\n            \\"partial_r\\": _num(self.partial_r),\\n            \\"ci_low\\": _num(self.ci_low),\\n            \\"ci_high\\": _num(self.ci_high),\\n            \\"n\\": int(self.n),\\n            \\"p_boot\\": _num(self.p_boot),\\n            \\"q_value\\": _num(self.q_value),\\n            \\"delta_r2\\": _num(self.delta_r2),\\n            \\"r2_base\\": _num(self.r2_base),\\n            \\"r2_full\\": _num(self.r2_full),\\n            \\"reliability\\": _num(self.reliability),\\n            \\"leave_one_out\\": {k: _num(v) for k, v in self.loo.items()},\\n            \\"ols\\": self.ols,\\n            **self.extras,\\n        }\\n        return payload\\n\\n    def sentence(self) -> str:\\n        \\"\\"\\"A single defensible claim, with uncertainty and incremental value.\\"\\"\\"\\n        sign = \\"positively\\" if self.partial_r > 0 else \\"negatively\\"\\n        return (\\n            f\\"In {self.position_group}, {self.trait} is {sign} associated with \\"\\n            f\\"{self.outcome} after controlling for the conventional Combine battery \\"\\n            f\\"and draft capital (partial r = {self.partial_r:+.2f}, \\"\\n            f\\"95% CI [{self.ci_low:+.2f}, {self.ci_high:+.2f}], n = {self.n}), \\"\\n            f\\"adding {self.delta_r2:+.3f} out-of-sample R^2.\\"\\n        )\\n\\n\\ndef _num(v: Any) -> Optional[float]:\\n    try:\\n        f = float(v)\\n    except (TypeError, ValueError):\\n        return None\\n    return None if not np.isfinite(f) else f\\n\\n\\n# ---------------------------------------------------------------------------\\n# Confirmation of a single linkage\\n# ---------------------------------------------------------------------------\\ndef confirm_linkage(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    outcome: str,\\n    trait: str,\\n    *,\\n    n_boot: int = 4000,\\n    seed: int = 0,\\n    cluster_by: Optional[str] = \\"draft_year\\",\\n) -> LinkageResult:\\n    \\"\\"\\"Full confirmatory battery for one (position group, trait, outcome) triple.\\"\\"\\"\\n    Z, y, sub, _ = feasibility.build_design(analysis, position_group, outcome)\\n    Z = Z.apply(feasibility._winsorize)\\n    x = feasibility._winsorize(pd.to_numeric(sub[trait], errors=\\"coerce\\"))\\n\\n    boot = stats.bootstrap_partial_correlation(\\n        x, y, Z, n_boot=n_boot, seed=seed,\\n        cluster=sub[cluster_by] if cluster_by and cluster_by in sub.columns else None,\\n    )\\n    cv = feasibility.cv_incremental_r2(Z, x, y)\\n\\n    # leave-one-out influence on the partial correlation\\n    frame = pd.concat([x.rename(\\"x\\"), y.rename(\\"y\\"), sub[[\\"nfl_id\\"]], Z], axis=1)\\n    loo_frame = stats.leave_one_out_stability(frame[\\"x\\"], frame[\\"y\\"], Z)\\n    loo = stats.influence_summary(loo_frame)\\n\\n    # a robust-regression view of the effect size in outcome standard deviations\\n    ols_text = None\\n    try:\\n        design = Z.copy()\\n        design[\\"__trait_z\\"] = stats.zscore_within(x)\\n        fitted = stats.ols_robust(design, stats.zscore_within(y))\\n        ols_text = fitted.summary()\\n    except Exception as exc:  # pragma: no cover\\n        log.warning(\\"OLS failed for %s/%s: %s\\", trait, outcome, exc)\\n\\n    # reliability of the trait across the player\'s own repeated attempts\\n    reliability = None\\n    try:\\n        attempts = io.read_parquet_cache(\\"combine_attempt_features.parquet\\")\\n        if attempts is None:\\n            raise FileNotFoundError(\\"combine attempt features not cached\\")\\n        drill, feature, _ = feasibility.parse_trait(trait)\\n        drill_type = {\\n            \\"three_cone_drill\\": \\"THREE_CONE_DRILL\\",\\n            \\"short_shuttle\\": \\"SHORT_SHUTTLE\\",\\n            \\"forty_yard_dash\\": \\"FORTY_YARD_DASH\\",\\n        }.get(drill)\\n        if drill_type and feature in attempts.columns:\\n            rel = feasibility.split_half_reliability(\\n                attempts, feature=feature, drill_type=drill_type\\n            )\\n            reliability = rel.get(\\"reliability\\")\\n        elif drill == \\"all_drills\\" or trait.startswith(\\"pooled_\\"):\\n            # pooled traits are averages over many attempts and drills, so their\\n            # reliability is estimated directly with a bootstrap split-half\\n            reliability = _pooled_reliability(attempts, trait)\\n    except FileNotFoundError:\\n        pass\\n\\n    return LinkageResult(\\n        position_group=position_group,\\n        outcome=outcome,\\n        trait=trait,\\n        partial_r=boot[\\"estimate\\"],\\n        ci_low=boot[\\"ci_low\\"],\\n        ci_high=boot[\\"ci_high\\"],\\n        n=int(boot[\\"n\\"]),\\n        p_boot=boot[\\"p_boot\\"],\\n        q_value=float(\\"nan\\"),\\n        delta_r2=cv.get(\\"delta_r2\\", float(\\"nan\\")),\\n        r2_base=cv.get(\\"r2_base\\", float(\\"nan\\")),\\n        r2_full=cv.get(\\"r2_full\\", float(\\"nan\\")),\\n        reliability=reliability,\\n        loo=loo,\\n        ols=ols_text,\\n    )\\n\\n\\n# ---------------------------------------------------------------------------\\n# From trait to published metric\\n# ---------------------------------------------------------------------------\\nDEFAULT_METRIC_NAME = \\"Break Retention Index\\"\\n\\n\\ndef build_metric(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    *,\\n    retention_traits: Sequence[str],\\n    regain_traits: Sequence[str] = (),\\n    name: str = DEFAULT_METRIC_NAME,\\n) -> pd.DataFrame:\\n    \\"\\"\\"Construct the composite metric and attach it to the analysis rows.\\n\\n    Components are standardised *within the position group* before averaging so\\n    that a receiver and a defensive tackle are never compared on a raw quantity\\n    that means different things for each.\\n\\n    ``retention_traits`` enter with a positive sign (keep more of your speed\\n    through the break is good); ``regain_traits`` enter with a negative sign\\n    (taking longer to get back up to speed is bad).\\n    \\"\\"\\"\\n    components = list(retention_traits) + list(regain_traits)\\n    direction = [1.0] * len(retention_traits) + [-1.0] * len(regain_traits)\\n    sub = analysis[analysis[\\"position_group\\"] == position_group].copy()\\n    available = [c for c in components if c in sub.columns]\\n\\n    if not available:\\n        sub[name] = np.nan\\n        return sub\\n\\n    # keep at least one of each family available\\n    keep_ret = [c for c in retention_traits if c in available]\\n    keep_reg = [c for c in regain_traits if c in available]\\n    components = keep_ret + keep_reg\\n    direction = [1.0] * len(keep_ret) + [-1.0] * len(keep_reg)\\n\\n    # inverse-variance-free, equal-weight composite; weights are deliberately not\\n    # fitted to the outcome, which would make the metric circular\\n    sub[name] = stats.composite_metric(\\n        sub, components, direction=direction, group=\\"position_group\\"\\n    )\\n    sub[\\"metric_components\\"] = \\", \\".join(components)\\n    return sub\\n\\n\\ndef metric_validation(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    outcome: str,\\n    *,\\n    metric_name: str = DEFAULT_METRIC_NAME,\\n    n_boot: int = 3000,\\n    seed: int = 0,\\n) -> Dict[str, Any]:\\n    \\"\\"\\"Does the composite metric beat the stopwatch on the chosen outcome?\\"\\"\\"\\n    Z, y, sub, _ = feasibility.build_design(analysis, position_group, outcome)\\n    Z = Z.apply(feasibility._winsorize)\\n    if metric_name not in sub.columns:\\n        return {\\"error\\": f\\"metric {metric_name!r} not built\\"}\\n\\n    x = feasibility._winsorize(pd.to_numeric(sub[metric_name], errors=\\"coerce\\"))\\n    boot = stats.bootstrap_partial_correlation(x, y, Z, n_boot=n_boot, seed=seed)\\n    cv = feasibility.cv_incremental_r2(Z, x, y)\\n\\n    result: Dict[str, Any] = {\\n        \\"position_group\\": position_group,\\n        \\"outcome\\": outcome,\\n        \\"metric\\": metric_name,\\n        **{k: _num(v) for k, v in boot.items()},\\n        **{k: _num(v) for k, v in cv.items()},\\n    }\\n\\n    # Publish the components the metric was actually averaged from, so that any\\n    # prose describing the formula can be generated from them rather than\\n    # hard-coded (a hard-coded formula previously drifted away from the\\n    # components ``build_metric`` selected). Deduplicated because the selected\\n    # trait is frequently also its own top supporting trait.\\n    if \\"metric_components\\" in sub.columns and len(sub):\\n        components = str(sub[\\"metric_components\\"].iloc[0]).strip()\\n        if components:\\n            result[\\"metric_components\\"] = list(\\n                dict.fromkeys(c for c in components.split(\\", \\") if c)\\n            )\\n\\n    # rank stability of the published top-10 under bootstrap resampling\\n    try:\\n        stability = stats.rank_stability(sub, lambda s: s[metric_name])\\n    except Exception as exc:  # pragma: no cover\\n        log.warning(\\"rank stability failed: %s\\", exc)\\n        stability = {}\\n    result[\\"rank_stability_top10\\"] = stability\\n    return result\\n\\n\\ndef metric_gain_over_stopwatch(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    outcome: str,\\n    *,\\n    metric_name: str = DEFAULT_METRIC_NAME,\\n) -> Dict[str, float]:\\n    \\"\\"\\"Incremental R^2 of the metric on top of the *best* single traditional test.\\n\\n    This is the fairest possible comparison and the one a sceptical scout would\\n    demand: the metric must beat the single best stopwatch number, not merely the\\n    whole battery at once.\\n    \\"\\"\\"\\n    Z, y, sub, _ = feasibility.build_design(analysis, position_group, outcome)\\n    if metric_name not in sub.columns:\\n        return {}\\n    from sklearn.linear_model import RidgeCV\\n    from sklearn.model_selection import RepeatedKFold, cross_validate\\n    from sklearn.pipeline import make_pipeline\\n    from sklearn.preprocessing import StandardScaler\\n\\n    alphas = np.logspace(-2, 3, 24)\\n    cv = RepeatedKFold(n_splits=5, n_repeats=4, random_state=0)\\n\\n    def _r2(frame: pd.DataFrame) -> float:\\n        \\"\\"\\"CV R^2 of a single-feature model on its own complete cases.\\"\\"\\"\\n        if len(frame) < feasibility.MIN_N:\\n            return np.nan\\n        model = make_pipeline(StandardScaler(), RidgeCV(alphas=alphas))\\n        scores = cross_validate(\\n            model, frame[[\\"x\\"]].to_numpy(), frame[\\"y\\"].to_numpy(),\\n            cv=cv, scoring=\\"r2\\", error_score=np.nan,\\n        )[\\"test_score\\"]\\n        return float(np.nanmean(scores))\\n\\n    tests = [c for c in feasibility.POSITION_BASELINE_TESTS.get(position_group, [])\\n             if c in sub.columns]\\n    best_test, best_r2 = None, -np.inf\\n    for test in tests:\\n        t = feasibility._winsorize(pd.to_numeric(sub[test], errors=\\"coerce\\"))\\n        frame = pd.concat([t.rename(\\"x\\"), y.rename(\\"y\\")], axis=1).dropna()\\n        r2 = _r2(frame)\\n        if np.isfinite(r2) and r2 > best_r2:\\n            best_test, best_r2 = test, r2\\n\\n    x = feasibility._winsorize(pd.to_numeric(sub[metric_name], errors=\\"coerce\\"))\\n    metric_frame = pd.concat([x.rename(\\"x\\"), y.rename(\\"y\\")], axis=1).dropna()\\n    metric_r2 = _r2(metric_frame)\\n    return {\\n        \\"metric_n\\": int(len(metric_frame)),\\n        \\"best_traditional_test\\": best_test,\\n        \\"best_traditional_r2\\": _num(best_r2),\\n        \\"metric_r2\\": _num(metric_r2),\\n        \\"gain_over_best_test\\": (\\n            _num(metric_r2 - best_r2)\\n            if np.isfinite(best_r2) and np.isfinite(metric_r2)\\n            else None\\n        ),\\n    }\\n\\n\\n# ---------------------------------------------------------------------------\\n# Scouting blind spots\\n# ---------------------------------------------------------------------------\\ndef blind_spot_analysis(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    outcome: str,\\n    *,\\n    metric_name: str = DEFAULT_METRIC_NAME,\\n    traditional: Optional[str] = None,\\n    top: int = 10,\\n) -> Dict[str, Any]:\\n    \\"\\"\\"Players the sensor likes and the stopwatch does not (and the reverse).\\n\\n    This is the most actionable output a front office can use: a named list of\\n    prospects whose conventional grade and movement profile disagree, plus the\\n    in-game outcome that arbitrates between them.\\n    \\"\\"\\"\\n    Z, y, sub, _ = feasibility.build_design(analysis, position_group, outcome)\\n    if metric_name not in sub.columns or not traditional:\\n        return {\\"error\\": \\"metric or traditional column missing\\"}\\n\\n    quad = stats.quadrant_split(\\n        sub.assign(**{\\"__y\\": y}),\\n        sub[metric_name],\\n        pd.to_numeric(sub[traditional], errors=\\"coerce\\")\\n        if traditional in sub.columns\\n        else pd.Series(np.nan, index=sub.index),\\n        sensor_label=\\"sensor\\",\\n        traditional_label=\\"stopwatch\\",\\n    )\\n    if quad.empty:\\n        return {\\"error\\": \\"no overlapping observations\\"}\\n\\n    # does the in-game outcome actually favour the sensor\'s preferred quadrant?\\n    sensor_beats = quad[quad[\\"archetype\\"] == \\"sensor_beats_stopwatch\\"][\\"__y\\"]\\n    stopwatch_beats = quad[quad[\\"archetype\\"] == \\"stopwatch_beats_sensor\\"][\\"__y\\"]\\n    confirmed = quad[quad[\\"archetype\\"] == \\"confirmed_elite\\"][\\"__y\\"]\\n    limited = quad[quad[\\"archetype\\"] == \\"confirmed_limited\\"][\\"__y\\"]\\n\\n    # lower is better for some outcomes (e.g. get_off, pressure allowed)\\n    lower_is_better = outcome in {\\n        \\"pressure_allowed_rate\\", \\"sack_allowed_rate\\", \\"ttp_allowed_mean\\",\\n        \\"peak_pressure_prob_allowed\\", \\"get_off_mean\\", \\"get_off_p25\\",\\n        \\"time_to_pressure_mean\\", \\"epa_allowed_mean\\",\\n    }\\n\\n    def _mean(s: pd.Series) -> Optional[float]:\\n        return _num(s.mean()) if len(s) else None\\n\\n    verdict = None\\n    if _mean(sensor_beats) is not None and _mean(stopwatch_beats) is not None:\\n        better = (\\n            _mean(sensor_beats) > _mean(stopwatch_beats)\\n            if not lower_is_better\\n            else _mean(sensor_beats) < _mean(stopwatch_beats)\\n        )\\n        verdict = (\\n            \\"the in-game outcome favours the sensor\'s preferred quadrant\\"\\n            if better\\n            else \\"the in-game outcome does NOT favour the sensor\'s preferred quadrant\\"\\n        )\\n\\n    cols = [c for c in [\\"nfl_id\\", \\"display_name\\", \\"draft_overall_pick\\",\\n                        \\"draft_round\\", \\"archetype\\", \\"sensor_z\\", \\"trad_z\\",\\n                        \\"disagreement\\", \\"__y\\"] if c in quad.columns]\\n    return {\\n        \\"n\\": int(len(quad)),\\n        \\"archetype_counts\\": quad[\\"archetype\\"].value_counts().to_dict(),\\n        \\"outcome_mean_by_archetype\\": {\\n            k: _mean(v) for k, v in quad.groupby(\\"archetype\\")[\\"__y\\"]\\n        },\\n        \\"outcome\\": outcome,\\n        \\"lower_is_better\\": lower_is_better,\\n        \\"verdict\\": verdict,\\n        \\"hidden_talent\\": quad[quad[\\"archetype\\"] == \\"sensor_beats_stopwatch\\"]\\n        .nlargest(top, \\"disagreement\\")[cols]\\n        .to_dict(orient=\\"records\\"),\\n        \\"over_graded\\": quad[quad[\\"archetype\\"] == \\"stopwatch_beats_sensor\\"]\\n        .nsmallest(top, \\"disagreement\\")[cols]\\n        .to_dict(orient=\\"records\\"),\\n    }\\n\\n\\n# ---------------------------------------------------------------------------\\n# Driver\\n# ---------------------------------------------------------------------------\\ndef run_deep_dive(\\n    analysis: pd.DataFrame,\\n    position_group: str,\\n    outcome: str,\\n    trait: str,\\n    *,\\n    metric_name: str = DEFAULT_METRIC_NAME,\\n    retention_traits: Sequence[str] = (),\\n    regain_traits: Sequence[str] = (),\\n    traditional: Optional[str] = None,\\n) -> Dict[str, Any]:\\n    \\"\\"\\"Confirm one linkage, build the metric, and validate both.\\"\\"\\"\\n    paths.ensure_dirs()\\n\\n    linkage = confirm_linkage(analysis, position_group, outcome, trait)\\n    payload: Dict[str, Any] = {\\"linkage\\": linkage.to_dict(),\\n                               \\"linkage_sentence\\": linkage.sentence()}\\n\\n    if retention_traits or regain_traits:\\n        enriched = build_metric(\\n            analysis,\\n            position_group,\\n            retention_traits=retention_traits or (trait,),\\n            regain_traits=regain_traits,\\n            name=metric_name,\\n        )\\n        payload[\\"metric_validation\\"] = metric_validation(\\n            enriched, position_group, outcome, metric_name=metric_name\\n        )\\n        payload[\\"metric_vs_stopwatch\\"] = metric_gain_over_stopwatch(\\n            enriched, position_group, outcome, metric_name=metric_name\\n        )\\n        payload[\\"blind_spots\\"] = blind_spot_analysis(\\n            enriched, position_group, outcome, metric_name=metric_name,\\n            traditional=traditional,\\n        )\\n        out_table = enriched[[\\n            c for c in [\\"nfl_id\\", \\"display_name\\", \\"position_group\\",\\n                        \\"draft_overall_pick\\", metric_name] if c in enriched.columns\\n        ]].sort_values(metric_name, ascending=False)\\n        out_table.to_csv(paths.REPORT_DIR / f\\"metric_leaderboard_{position_group}.csv\\",\\n                         index=False)\\n        payload[\\"leaderboard_top\\"] = out_table.head(15).to_dict(orient=\\"records\\")\\n\\n    (paths.REPORT_DIR / \\"deep_dive.json\\").write_text(\\n        json.dumps(payload, indent=2, default=str), encoding=\\"utf-8\\"\\n    )\\n    return payload\\n\\ndef _pooled_reliability(\\n    attempts: pd.DataFrame, trait: str, *, n_rep: int = 200, seed: int = 0\\n) -> Optional[float]:\\n    \\"\\"\\"Split-half reliability of a pooled trait across a player\'s own cuts.\\n\\n    Each player\'s detected cuts are split at random into two halves; the trait\\n    is recomputed on each half and the correlation across players is corrected\\n    with Spearman-Brown. This bounds how strongly the trait *could* correlate\\n    with anything, which is an essential honesty check for a null result: a\\n    trait with 0.3 reliability cannot show a large correlation no matter how\\n    real the underlying signal is.\\n    \\"\\"\\"\\n    from . import kinematics\\n\\n    if attempts.empty:\\n        return None\\n    rng = np.random.default_rng(seed)\\n    correlations: List[float] = []\\n    for _ in range(n_rep):\\n        shuf = attempts.sample(frac=1.0, random_state=int(rng.integers(1 << 31)))\\n        shuf = shuf.assign(_half=np.arange(len(shuf)) % 2)\\n        halves = []\\n        for half in (0, 1):\\n            part = shuf[shuf[\\"_half\\"] == half]\\n            pooled = kinematics.pooled_cod_player_features(part)\\n            col = trait if trait in pooled.columns else None\\n            if col is None:\\n                return None\\n            halves.append(pooled.set_index(\\"nfl_id\\")[col])\\n        joined = pd.concat(halves, axis=1).dropna()\\n        if len(joined) < 20:\\n            continue\\n        r = joined.corr().iloc[0, 1]\\n        if np.isfinite(r) and r > -1:\\n            correlations.append(2 * r / (1 + r))\\n    return float(np.mean(correlations)) if correlations else None\\n"}'
_BDB_SOURCES = _json.loads(_BDB_PAYLOAD)

_pkg = types.ModuleType('bdb')
_pkg.__path__ = []
sys.modules['bdb'] = _pkg
for _name, _src in _BDB_SOURCES.items():
    _mod = types.ModuleType('bdb.' + _name)
    _mod.__package__ = 'bdb'
    # the modules resolve their own paths from __file__, so give them one
    _mod.__file__ = f'<bdb.{_name}>'
    setattr(_pkg, _name, _mod)
    sys.modules['bdb.' + _name] = _mod
for _name in _BDB_SOURCES:
    exec(compile(_BDB_SOURCES[_name], f'<bdb.{_name}>', 'exec'),
         sys.modules['bdb.' + _name].__dict__)
print('inlined bdb modules:', ', '.join(_BDB_SOURCES))


In [ ]:
# Locate the competition CSVs. Nothing needs editing on Kaggle or locally.
import os
from pathlib import Path
from bdb import paths

def _find_data():
    if paths.missing_files() == []:
        return paths.DATA_DIR
    cands = []
    here = Path.cwd()
    cands += [here / 'multi_agents/competition/nfl-big-data-bowl-2027/data',
              here / 'multi_agents/competition/nfl_big_data_bowl_2027/data']
    for root in [Path('/kaggle/input')]:
        if root.is_dir():
            cands += sorted(root.glob('*')) + sorted(root.glob('*/*'))
    for c in cands:
        if c.is_dir() and (c / 'players.csv').exists() and (c / 'combine_tracking.csv').exists():
            return c
    return None

_data = _find_data()
if _data is None:
    raise SystemExit('Competition CSVs not found. Attach the nfl-big-data-bowl-2027 dataset.')
paths.DATA_DIR = _data

# Derived artefacts go to a writable directory (Kaggle input is read-only).
from pathlib import Path as _P
_out = _P('/kaggle/working/bdb_out') if _P('/kaggle/working').is_dir() else _P.cwd() / 'bdb_out'
_out.mkdir(parents=True, exist_ok=True)
paths.ARTIFACT_DIR = _out
paths.CACHE_DIR = _out / 'cache'
paths.FIGURE_DIR = _out / 'figures'
paths.REPORT_DIR = _out / 'reports'
paths.ensure_dirs()
print('data   :', paths.DATA_DIR)
print('outputs:', paths.ARTIFACT_DIR)
print('missing:', paths.missing_files() or 'none')


## 1. Build the 10 Hz sensor features from the raw Combine tracking


In [ ]:
import pandas as pd
from bdb import io, kinematics
pd.set_option('display.width', 160)

tracking = io.load_combine_tracking(players_only=True)
attempts = kinematics.extract_all_attempt_features(tracking)
player_feats = kinematics.player_level_combine_features(attempts)
print(tracking.shape, '->', attempts.shape, '->', player_feats.shape)


Each row of `attempts` is one drill attempt summarised into coach-readable movement traits: peak speed, entry speed into the sharpest cut, the minimum speed carried through it, path geometry, and the braking/re-acceleration shape.


## 2. Reliability screen (uses no outcome data)


In [ ]:
from bdb import reliability
rel = reliability.build_reliability_table(attempts)
print('traits scored:', len(rel))
rel[['feature', 'kind', 'reliability']].head(12)


Level and timing traits survive; ratio and difference traits mostly do not. This screen runs before any outcome is touched, and traits below the floor are dropped from the search space.


## 3. Build the in-game outcomes and the analysis table


In [ ]:
from bdb import ingame
ingame_feats = ingame.build_ingame_player_features()
table = io.build_analysis_table(player_feats, ingame_feats, force=True)
print(table.shape)


## 4. Hypothesis screen with FDR control


In [ ]:
from bdb import feasibility
feas = feasibility.scan_all(table)
print('pairs screened:', len(feas))
print('survivors     :', int(feas['adds_beyond_stopwatch'].sum()))
feas.sort_values('delta_r2', ascending=False).head(10)[
    ['position_group', 'outcome', 'trait', 'n', 'partial_spearman',
     'q_value', 'delta_r2']]


## 5. Confirm the selected linkage with bootstrap uncertainty


In [ ]:
from bdb import deepdive
result = deepdive.confirm_linkage(table, 'DL', 'sack_rate', 'skill_dl__entry_speed_max')
print(result.sentence())
print()
print(result.ols)


The claim is an *incremental* one: every model's baseline is the conventional Combine battery (40, 10-yard split, 3-cone, short shuttle, vertical, broad jump, bench) plus log draft capital, age and body size, median-imputed with explicit 'did not test' indicators.


## 6. Leave-one-out influence: one player or a real effect?


In [ ]:
import pandas as pd
from bdb import stats
Z, y, sub, _ = feasibility.build_design(table, 'DL', 'sack_rate')
x = pd.to_numeric(sub['skill_dl__entry_speed_max'], errors='coerce')
fr = pd.concat([x.rename('x'), y.rename('y'), Z], axis=1).dropna()
loo = stats.leave_one_out_stability(fr['x'], fr['y'], Z)
stats.influence_summary(loo)


---
### Reproducibility
This notebook is self-contained: the analysis package is inlined above and the competition data is located automatically. In the repository the identical code lives in `solutions/nfl_big_data_bowl_2027`, where `python -m bdb.cli all` rebuilds the features and the screen and `python -m bdb.cli deepdive` reproduces the confirmatory analysis and every figure.
